# LLM & Diffusion Model Inference Optimization

---

Modern generative AI models are large, and memory-hungry. This course covers the principal techniques used in production to make them faster — without retraining and with minimal accuracy loss.

**Learning Objectives:** by the end of this course you will be able to:
- Apply Post-Training Quantization (PTQ) to LLMs using `nvidia-modelopt`
- Measure and reduce KV cache memory with FP8 compression
- Accelerate diffusion model inference with training-free step caching
- Run AutoQuantization to find the optimal per-layer precision automatically
- Combine multiple techniques and reason about their compounding effects

**Prerequisites:** PyTorch fundamentals, transformer architecture basics, familiarity with diffusion models.

**Hardware:** H100 or A100 GPU (A100 used in this notebook — see notes on simulation mode).

**Software stack:** `nvidia-modelopt`, `diffusers`, `transformers`, running inside `nvcr.io/nvidia/pytorch:26.02-py3`.

---

### Contents

- **[Module 0: Setup & GPU Profiling](#Module-0:-Setup-&-GPU-Profiling)**
- **[Module 1: Post-Training Quantization (PTQ) for LLMs](#Module-1:-Post-Training-Quantization-(PTQ)-for-LLMs)**
  - [1.1 Baseline LLM](#1.1-Baseline-LLM)
  - [1.2 FP8 Min-Max Quantization](#1.2-FP8-Min-Max-Quantization)
  - [1.3 INT4 AWQ Quantization](#1.3-INT4-AWQ-Quantization)
  - [1.4 SmoothQuant INT8](#1.4-SmoothQuant-INT8)
  - [1.5 Comparison Table](#1.5-Comparison-Table)
  - [1.6 Export Quantized Checkpoint](#1.6-Export-Quantized-Checkpoint)
- **[Module 2: KV Cache Compression](#Module-2:-KV-Cache-Compression)**
  - [2.1 KV Cache Size Analysis](#2.1-KV-Cache-Size-Analysis)
  - [2.2 FP8 KV Cache Quantization](#2.2-FP8-KV-Cache-Quantization)
  - [2.3 Memory vs Sequence Length Plot](#2.3-Memory-vs-Sequence-Length-Plot)
- **[Module 3: Cache Diffusion](#Module-3:-Cache-Diffusion-(Denoising-Step-Caching))**
  - [3.1 Load SDXL](#3.1-Load-SDXL)
  - [3.2 Baseline Benchmark](#3.2-Baseline-Benchmark)
  - [3.3 Apply Cache Diffusion](#3.3-Apply-Cache-Diffusion)
  - [3.4 Visual Comparison](#3.4-Visual-Comparison)
  - [3.5 Cache Diffusion vs Naive Step Reduction](#3.5-Cache-Diffusion-vs-Naive-Step-Reduction)
  - [3.6 PixArt-Alpha Demo](#3.6-PixArt-Alpha-Demo)
- **[Module 4: AutoQuantization & Mixed Precision](#Module-4:-AutoQuantization-&-Mixed-Precision)**
  - [4.1 AutoQuantize](#4.1-AutoQuantize)
  - [4.2 Inspect Per-Layer Decisions](#4.2-Inspect-Per-Layer-Decisions)
  - [4.3 Benchmark AutoQ vs Uniform Quantization](#4.3-Benchmark-AutoQ-vs-Uniform-Quantization)
- **[Module 5: Combined Pipeline & Z-Image Outlook](#Module-5:-Combined-Pipeline-&-Z-Image-Outlook)**
  - [5.1 Stacking Optimizations](#5.1-Stacking-Optimizations)
  - [5.2 Diffusion Model Quantization](#5.2-Diffusion-Model-Quantization-(Script-Based))
  - [5.3 Z-Image Optimization Outlook](#5.3-Z-Image-Optimization-Outlook)
  - [5.4 Course Summary](#5.4-Course-Summary)
- **[Appendix A: Why PTQ Speedup Requires H100+](#Appendix-A:-Why-PTQ-Speedup-Requires-H100+-and-a-Deployment-Backend)**
- **[Appendix B: Adding Z-Image Support to Model Optimizer](#Appendix-B:-Adding-Z-Image-Support-to-NVIDIA-Model-Optimizer)**


---
# Module 0: Setup & GPU Profiling


Before running any experiment we need a consistent, instrumented environment. This module sets up three things:

1. **Environment variables** — suppress apex/layernorm conflicts inside the PyTorch container
2. **GPU diagnostics** — verify the GPU name, VRAM, and CUDA version are as expected
3. **Utility functions** — `gpu_mem_used()` and `free_memory()` that every later module reuses    to measure memory savings accurately

> **Expected output:** `nvidia-smi` should show an A100 or H100 with ≥ 40 GB VRAM. > If you see a smaller GPU, memory-intensive cells may OOM — check with your instructor.


In [ ]:
import os

# Required to avoid apex fused-layernorm conflicts inside the PyTorch container
os.environ["NVTE_APEX_FUSED_LAYERNORM"] = "0"
os.environ["APEX_FUSED_RMSNORM"] = "0"

print("Environment variables set.")

In [ ]:
import subprocess
import torch

result = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,memory.free,utilization.gpu",
     "--format=csv,noheader,nounits"],
    capture_output=True, text=True
)
print("GPU Info (nvidia-smi):")
for i, line in enumerate(result.stdout.strip().split("\n")):
    name, total, free, util = [x.strip() for x in line.split(",")]
    # Some hardware (e.g. DGX Spark GB10) reports [N/A] for memory queries
    total_str = f"{int(total):,} MB" if total not in ("[N/A]", "N/A") else "N/A"
    free_str  = f"{int(free):,} MB"  if free  not in ("[N/A]", "N/A") else "N/A"
    print(f"  GPU {i}: {name}  |  Total: {total_str}  |  Free: {free_str}  |  Util: {util}%")

print(f"\nPyTorch : {torch.__version__}")
print(f"CUDA    : {torch.version.cuda}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU     : {props.name}")
    print(f"VRAM    : {props.total_memory / 1e9:.1f} GB")

In [ ]:
import time, gc
import torch

def gpu_mem_used(device=0):
    """Return GPU memory currently allocated (MB)."""
    torch.cuda.synchronize(device)
    return torch.cuda.memory_allocated(device) / 1e6

def benchmark_inference(fn, n=10, warmup=3):
    """
    Benchmark a zero-argument callable.
    Returns (mean_ms, std_ms) using CUDA events for accurate GPU timing.
    """
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()
    start = torch.cuda.Event(enable_timing=True)
    end   = torch.cuda.Event(enable_timing=True)
    times = []
    for _ in range(n):
        start.record(); fn(); end.record()
        torch.cuda.synchronize()
        times.append(start.elapsed_time(end))
    mean_ms = sum(times) / len(times)
    std_ms  = (sum((t - mean_ms)**2 for t in times) / len(times)) ** 0.5
    return mean_ms, std_ms

def free_memory():
    gc.collect()
    torch.cuda.empty_cache()

print(f"Baseline GPU memory in use: {gpu_mem_used():.1f} MB")
print("Utilities ready: gpu_mem_used(), benchmark_inference(), free_memory()")

<h2 style="color: green;">✓ Module 0 Complete</h2>

Environment is configured and GPU diagnostics look good. The `gpu_mem_used()` and `free_memory()` helpers are now available globally and will be used throughout the course to measure the effect of each optimization.


---
# Module 1: Post-Training Quantization (PTQ) for LLMs


## 1.0 Theory: Why Quantization?

A modern LLM like Llama-3-70B stores 70 billion parameters. In FP32 that is **280 GB** — more than
the combined VRAM of 3 H100s. BF16 halves this to 140 GB. Quantization takes it further.

As bit-width decreases, the representable range shrinks dramatically:

![Range and detail when quantizing from FP16 to FP8 or FP4](https://developer-blogs.nvidia.com/wp-content/uploads/2025/12/range-detail-quantizing-fp16-fp8-fp4-png.webp)

*FP16 can represent values up to ±65,504. FP4 is capped at ±6. Quantization is a precision–range
trade-off: fewer bits means faster compute and smaller memory, but tighter representable range.*

### Uniform Quantization

Maps floating-point values into a finite set of integers:

$$x_q = \text{clamp}\!\left(\text{round}\!\left(\frac{x}{s}\right) + z,\; 0,\; 2^b - 1\right)$$

where $s$ is the **scale factor**, $z$ is the **zero-point**, and $b$ is the bit-width.  
Dequantization: $\hat{x} = s(x_q - z)$.

### PTQ vs QAT

- **PTQ:** quantize a pretrained model with a small calibration dataset. No gradient updates. Fast (< 1 hour).
- **QAT:** simulate quantization during fine-tuning using the Straight-Through Estimator (STE) for gradient flow. More accurate for < 4 bits, but requires training.

### Granularity

| Granularity | Scale per... | Memory overhead | Accuracy |
|---|---|---|---|
| Per-tensor | whole tensor | minimal | lowest |
| Per-channel | output channel | small | good |
| Per-group (block) | group of $g=128$ elements | moderate | best for sub-4-bit |

### Format Comparison

| Format | Bits | Compression vs BF16 | HW support | Notes |
|--------|------|-----|---|---|
| BF16 | 16 | 1× (baseline) | All GPUs | Training default |
| INT8 | 8 | 2× | Ampere+ (INT8 GEMM) | SmoothQuant needed for LLMs |
| FP8 (E4M3) | 8 | 2× | Hopper/Ada+ (≥ SM 8.9) | Better dynamic range than INT8 |
| INT4-AWQ | 4 | 4× | All (fused dequant kernels) | AWQ scaling preserves accuracy |
| NVFP4 | 4 | 4× | Blackwell (B100/B200) | Next-gen, highest throughput |

### Accuracy Impact (from ModelOpt benchmarks on MMLU)

| Format | MMLU Score Loss | Notes |
|--------|----------------|-------|
| FP8 | 0.21 – 0.87% | Near-lossless, production standard on H100 |
| INT4-AWQ | 0.85 – 2.11% | Acceptable for most tasks |
| INT8 SmoothQuant | 1.47 – 2.75% | Higher loss; use when FP8 HW unavailable |

### SmoothQuant

Addresses the outlier activation problem by migrating quantization difficulty from activations to weights:

$$Y = (X \cdot \text{diag}(s)^{-1}) \cdot (\text{diag}(s) \cdot W)$$

### AWQ

Protects the most important weight channels (highest activation magnitude) by scaling them before
quantization. The protected channels carry the most information and dominate accuracy.

> **Key Takeaway:** FP8 is the production sweet spot on H100: near-lossless accuracy with 2× throughput
> gain **when deployed via TRT-LLM or vLLM**. INT4-AWQ is used when fitting large models on a single
> GPU is the primary constraint.

### 📚 Literature

- **SmoothQuant** — Xiao et al., ICML 2023. Activation-weight migration for INT8 LLM quantization.  
  → [arXiv:2211.10438](https://arxiv.org/abs/2211.10438)
- **AWQ** — Lin et al., MLSys 2024 *(Best Paper)*. Activation-aware weight quantization.  
  → [arXiv:2306.00978](https://arxiv.org/abs/2306.00978)
- **GPTQ** — Frantar et al., ICLR 2023. Second-order error minimisation for INT4 PTQ.  
  → [arXiv:2210.17323](https://arxiv.org/abs/2210.17323)
- **FP8 Formats** — Micikevicius et al., 2022. Industry specification for E4M3/E5M2 formats.  
  → [arXiv:2209.05433](https://arxiv.org/abs/2209.05433)


## 1.1 Baseline LLM

We use **Qwen2.5-1.5B-Instruct** as our reference model throughout Module 1. It is small enough for fast classroom demos (~3 GB in BF16) but architecturally representative of modern decoder-only transformers:

- **Grouped-Query Attention (GQA):** fewer KV heads than query heads → native KV memory savings
- **SwiGLU FFN:** gated activation function, standard in Llama-3 / Qwen / Mistral families
- **RoPE positional encoding:** relative position without learned embeddings
- **Tied input/output embeddings:** the most quantization-sensitive component

We load the model in BF16 (the training default on A100/H100), time a single generation pass, and record memory usage. This is the baseline every later section compares against.

> ⏱️ **Expected runtime:** ~30 s for model load, ~2 s per generation.


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

print("Loading tokenizer ...")
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

print("Loading model in BF16 ...")
free_memory()
mem_before = gpu_mem_used()

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="cuda",
    trust_remote_code=True,
).eval()

model_mem_mb = gpu_mem_used() - mem_before
n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters  : {n_params/1e9:.2f}B")
print(f"GPU memory  : {model_mem_mb:.0f} MB")
print(f"dtype       : {next(model.parameters()).dtype}")

In [ ]:
prompt = "Explain the concept of neural network quantization in simple terms."
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

def run_baseline():
    with torch.no_grad():
        return model.generate(
            **inputs, max_new_tokens=100, do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

baseline_ms, baseline_std = benchmark_inference(run_baseline, n=5, warmup=2)
baseline_tokens_sec = 100 / (baseline_ms / 1000)

print(f"BF16 Baseline:")
print(f"  Latency (100 tokens): {baseline_ms:.1f} +/- {baseline_std:.1f} ms")
print(f"  Throughput          : {baseline_tokens_sec:.1f} tokens/sec")
print(f"  GPU memory (model)  : {model_mem_mb:.0f} MB")

results = {"BF16 (baseline)": {"mem_mb": model_mem_mb, "latency_ms": baseline_ms, "tokens_sec": baseline_tokens_sec}}

## 1.2 FP8 Min-Max Quantization

The `modelopt` PTQ API is three steps:
1. Build a calibration dataloader
2. Create a `forward_loop`
3. Call `mtq.quantize(model, config, forward_loop)`

`mtq.quantize` inserts `TensorQuantizer` wrappers around weight and activation tensors,
runs the forward loop to collect per-channel min/max statistics, then freezes the scale factors.

**FP8 format (E4M3):** 1 sign bit, 4 exponent bits, 3 mantissa bits. Representable range ±448.
Specified in the joint NVIDIA/Intel/ARM standard [arXiv:2209.05433](https://arxiv.org/abs/2209.05433).

> ⚠️ **Simulation mode:** `mtq.quantize()` runs quantize→dequantize→BF16 GEMM, which is
> *slower* than baseline BF16. Real FP8 speedup requires deployment via TRT-LLM or vLLM on H100+.
> See Appendix A for a detailed explanation.


In [ ]:
import modelopt.torch.quantization as mtq
from modelopt.torch.utils.dataset_utils import create_forward_loop, get_dataset_dataloader

print("Building calibration dataloader (cnn_dailymail, 64 samples) ...")
dataloader = get_dataset_dataloader(
    dataset_name="cnn_dailymail",
    tokenizer=tokenizer,
    batch_size=4,
    num_samples=64,
    device="cuda",
)
forward_loop = create_forward_loop(dataloader=dataloader)
print(f"Dataloader ready: {len(dataloader)} batches of 4")

In [ ]:
print("Applying FP8 min-max quantization ...")
model_fp8 = mtq.quantize(model, mtq.FP8_DEFAULT_CFG, forward_loop=forward_loop)

fp8_mem_mb = gpu_mem_used()
print("FP8 quantization complete.")
print()
mtq.print_quant_summary(model_fp8)

In [ ]:
def run_fp8():
    with torch.no_grad():
        return model_fp8.generate(
            **inputs, max_new_tokens=100, do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

fp8_ms, fp8_std = benchmark_inference(run_fp8, n=5, warmup=2)
fp8_tokens_sec = 100 / (fp8_ms / 1000)

print(f"FP8: {fp8_ms:.1f} ms  |  {fp8_tokens_sec:.1f} tok/s  |  {fp8_mem_mb:.0f} MB  |  {baseline_ms/fp8_ms:.2f}x speedup")
results["FP8"] = {"mem_mb": fp8_mem_mb, "latency_ms": fp8_ms, "tokens_sec": fp8_tokens_sec}

## 1.3 INT4 AWQ Quantization

**AWQ (Activation-aware Weight Quantization)** [[arXiv:2306.00978](https://arxiv.org/abs/2306.00978)]
is the state-of-the-art for 4-bit weight-only quantization.
Weights are stored as INT4 but dequantized to BF16 before GEMM.
With group size $g=128$, AWQ stores scale factors per group — about 1% overhead vs the weights.

**Key insight:** not all weight channels matter equally. AWQ identifies the 1% of channels with
the highest activation magnitudes (the "salient" channels) and protects them with larger scales
before quantization, preserving accuracy at minimal cost.

This trades compute for memory bandwidth: ideal when the model does not fit in VRAM at higher precision.

> ⚠️ Simulation mode applies here too — expect slower-than-BF16 latency in PyTorch.


In [ ]:
del model_fp8
free_memory()

model_fresh = AutoModelForCausalLM.from_pretrained(
    model_name, torch_dtype=torch.bfloat16, device_map="cuda", trust_remote_code=True
).eval()

print("Applying INT4-AWQ quantization ...")
model_int4 = mtq.quantize(model_fresh, mtq.INT4_AWQ_CFG, forward_loop=forward_loop)
int4_mem_mb = gpu_mem_used()
print("INT4-AWQ complete.")

In [ ]:
def run_int4():
    with torch.no_grad():
        return model_int4.generate(
            **inputs, max_new_tokens=100, do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

int4_ms, int4_std = benchmark_inference(run_int4, n=5, warmup=2)
int4_tokens_sec = 100 / (int4_ms / 1000)
print(f"INT4-AWQ: {int4_ms:.1f} ms  |  {int4_tokens_sec:.1f} tok/s  |  {int4_mem_mb:.0f} MB  |  {baseline_ms/int4_ms:.2f}x speedup")
results["INT4-AWQ"] = {"mem_mb": int4_mem_mb, "latency_ms": int4_ms, "tokens_sec": int4_tokens_sec}

## 1.4 SmoothQuant INT8

SmoothQuant [[arXiv:2211.10438](https://arxiv.org/abs/2211.10438)] addresses the outlier
activation problem. LLMs produce activation tensors with extreme outliers (100× larger than
typical values) that make per-tensor quantization inaccurate.

For a linear layer $Y = XW$, SmoothQuant applies a mathematically lossless channel-wise transform:

$$Y = (X \cdot \text{diag}(s)^{-1}) \cdot (\text{diag}(s) \cdot W)$$

The per-channel scale $s$ equalizes variance between $X$ and $W$, chosen as:

$$s_j = \frac{\max(|X_j|)^\alpha}{\max(|W_j|)^{1-\alpha}}, \quad \alpha \in [0, 1]$$

The transform is lossless — accuracy degrades only from the subsequent INT8 quantization
of the smoothed tensors. $\alpha=0.5$ works well in practice.


In [ ]:
del model_int4
free_memory()

model_sq = AutoModelForCausalLM.from_pretrained(
    model_name, torch_dtype=torch.bfloat16, device_map="cuda", trust_remote_code=True
).eval()

print("Applying SmoothQuant INT8 quantization ...")
model_int8 = mtq.quantize(model_sq, mtq.INT8_SMOOTHQUANT_CFG, forward_loop=forward_loop)
int8_mem_mb = gpu_mem_used()
print("SmoothQuant INT8 complete.")

In [ ]:
def run_int8():
    with torch.no_grad():
        return model_int8.generate(
            **inputs, max_new_tokens=100, do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

int8_ms, int8_std = benchmark_inference(run_int8, n=5, warmup=2)
int8_tokens_sec = 100 / (int8_ms / 1000)
print(f"INT8-SQ: {int8_ms:.1f} ms  |  {int8_tokens_sec:.1f} tok/s  |  {int8_mem_mb:.0f} MB  |  {baseline_ms/int8_ms:.2f}x speedup")
results["INT8-SmoothQuant"] = {"mem_mb": int8_mem_mb, "latency_ms": int8_ms, "tokens_sec": int8_tokens_sec}

## 1.5 Comparison Table

> **Why is quantization slower here?** ModelOpt's `quantize()` inserts `TensorQuantizer`
> wrappers that **simulate** quantization (quantize → dequantize → standard GEMM). The actual
> throughput gains require **deployment** backends with native kernel support:
> - **TRT-LLM** — native FP8/INT4 GEMM kernels, 2–4× throughput gains on H100
> - **vLLM** — Marlin/GPTQ kernels for INT4-AWQ, FlashInfer for FP8 attention
> - **SGLang** — FP8 attention via FlashInfer
>
> See Appendix A for the full explanation with roofline analysis.

### Hardware–Format Compatibility

| GPU | Architecture | Best Format | Native FP8 GEMM |
|-----|---|---|:-:|
| A100 | Ampere | INT8 SmoothQuant or INT4-AWQ | ✗ |
| H100 / H200 | Hopper | **FP8** | ✓ |
| Ada (RTX 4090) | Ada Lovelace | **FP8** | ✓ |
| B200-300 / GB200-300 | Blackwell | **NVFP4 + FP8** | ✓ |


In [ ]:
import pandas as pd

bl_mem = results["BF16 (baseline)"]["mem_mb"]
bl_lat = results["BF16 (baseline)"]["latency_ms"]

rows = []
for name, r in results.items():
    rows.append({
        "Format": name,
        "GPU Memory (MB)": f"{r['mem_mb']:.0f}",
        "Mem Reduction": f"{bl_mem/r['mem_mb']:.2f}x",
        "Latency (ms)": f"{r['latency_ms']:.1f}",
        "Speedup": f"{bl_lat/r['latency_ms']:.2f}x",
        "Tokens/sec": f"{r['tokens_sec']:.1f}",
    })

df = pd.DataFrame(rows).set_index("Format")
try:
    from IPython.display import display; display(df)
except Exception:
    print(df.to_string())

## 1.6 Export Quantized Checkpoint

`export_hf_checkpoint` serializes the quantized model to HuggingFace safetensors format. The quantization scale factors are embedded as JSON in the file metadata under the key `_quantization_metadata`, making the checkpoint self-contained and portable.

**Deployment targets after export:**
- **vLLM** — load directly with `--quantization modelopt`; uses Marlin/GPTQ kernels for INT4,   FlashInfer for FP8 attention
- **SGLang** — same as vLLM; FP8 attention via FlashInfer
- **TRT-LLM** — convert via `trtllm-build`; native FP8 GEMM on H100

> 📁 The exported checkpoint is saved to `./exports/<format>/`. > It includes `config.json`, `*.safetensors`, and `tokenizer.*` files — ready to load with > `AutoModelForCausalLM.from_pretrained()` on any supported serving backend.


In [ ]:
from modelopt.torch.export import export_hf_checkpoint
import os, glob

del model_int8
free_memory()

model_export = AutoModelForCausalLM.from_pretrained(
    model_name, torch_dtype=torch.bfloat16, device_map="cuda", trust_remote_code=True
).eval()
model_export = mtq.quantize(model_export, mtq.FP8_DEFAULT_CFG, forward_loop=forward_loop)

export_dir = "exports/qwen-fp8/"
os.makedirs(export_dir, exist_ok=True)

print(f"Exporting to {export_dir} ...")
export_hf_checkpoint(model_export, export_dir=export_dir)

for f in sorted(glob.glob(os.path.join(export_dir, "*"))):
    print(f"  {os.path.basename(f):40s}  {os.path.getsize(f)/1e6:.1f} MB")

print("\nExport complete. _quantization_metadata is embedded in the safetensors header.")

<h2 style="color: green;">✓ Module 1 — Post-Training Quantization</h2>

**What you did:**
- Loaded Qwen2.5-1.5B in BF16 and established a memory + latency baseline
- Applied FP8, INT4-AWQ, and INT8 SmoothQuant quantization via `mtq.quantize()`
- Observed that simulation mode (PyTorch) adds overhead — real speedups need H100 + TRT-LLM/vLLM
- Exported a quantized checkpoint in HuggingFace safetensors format

**Key numbers from this run:**

| Format | Memory vs BF16 | Latency (simulation) |
|---|---|---|
| BF16 baseline | 1.0× | 1.0× |
| FP8 | ~0.5× | slower (simulation) |
| INT4-AWQ | ~0.25× | slower (simulation) |
| INT8 SmoothQuant | ~0.5× | slower (simulation) |

→ Next module: **KV Cache Compression** — attack the other major memory cost in LLM serving.


---
# Module 2: KV Cache Compression


## 2.0 Theory: The KV Cache Memory Problem

During autoregressive decoding, a transformer stores the Key and Value projections for every
previously generated token in every layer. This is the **KV cache**.

### Memory Formula

$$\text{KV cache} = 2 \times L \times H_{kv} \times d_{head} \times S \times B \times \text{bytes}$$

where $L$ = layers, $H_{kv}$ = KV heads, $d_{head}$ = head dim, $S$ = sequence length, $B$ = batch size.

**Example: Llama-3-70B at 8k context, batch=16, BF16:**
$$2 \times 80 \times 8 \times 128 \times 8192 \times 16 \times 2 \approx 42 \text{ GB}$$

This can easily exceed the weight memory and is the dominant factor in LLM serving throughput.

### PagedAttention

Modern LLM serving systems (vLLM [[arXiv:2309.06180](https://arxiv.org/abs/2309.06180)]) manage
KV cache in **fixed-size blocks** analogous to OS virtual memory pages. This eliminates memory
fragmentation and allows cross-request **prefix reuse**: if multiple requests share the same
prompt prefix, their KV blocks are stored once and referenced by all requests — reducing both
memory and compute.

### KV Cache Quantization

Quantizing the KV cache to FP8 halves the bytes read/written per attention operation.
The quantization formula applied per token:

$$q = \text{clamp}\!\left(\text{round}\!\left(\frac{x}{s}\right), -127, 127\right)$$

Dequantization happens on-the-fly inside the attention kernel — no extra memory traffic.

**Measured gains on H100 (from TensorRT-LLM benchmarks):**

| Configuration | Throughput | vs BF16 KV |
|---|---|---|
| BF16 KV cache | 3,389 tok/s | 1.0× |
| FP8 KV cache | 5,299 tok/s | **+56%** |
| FP8 KV + FP8 weights | > 5,299 tok/s | **> 2–3× larger batch** |

### Compression Strategies

| Strategy | Memory Reduction | Accuracy Impact |
|---|---|---|
| FP8 KV | 2× | Minimal (<0.1 PPL) |
| INT4 KV [[arXiv:2405.03917](https://arxiv.org/abs/2405.03917)] | 4× | Small |
| GQA/MQA (architectural) | $H_q / H_{kv}$× | None (trained with it) |
| StreamingLLM | Bounded memory | OK for long context |

> **Key Takeaway:** FP8 KV quantization is almost always free and gives 2× KV cache reduction,
> translating directly to 2× more batch size or 2× longer context on the same hardware.

### 📚 Literature

- **PagedAttention** — Kwon et al., SOSP 2023. Block-based KV memory management for LLM serving.  
  → [arXiv:2309.06180](https://arxiv.org/abs/2309.06180)
- **KV Cache is 1 Bit Per Channel** — Zhang et al., NeurIPS 2024. Extreme KV cache quantization.  
  → [arXiv:2405.03917](https://arxiv.org/abs/2405.03917)


## 2.1 KV Cache Size Analysis

Before compressing the KV cache, we need to understand its scale. The cell below computes the theoretical KV cache size in GB for a range of sequence lengths and batch sizes, using the formula from the theory section.

We compare three model families to show how architectural choices (number of KV heads, head dimension, layer count) dominate the memory budget:

| Model | Params | KV heads | Head dim | BF16 KV @ 4k context, batch 8 |
|---|---|---|---|---|
| Qwen2.5-1.5B | 1.5B | 2 | 128 | ~0.5 GB |
| Llama-3-8B | 8B | 8 | 128 | ~2.1 GB |
| Llama-3-70B | 70B | 8 | 128 | ~16.8 GB |

> **What to look for:** KV memory scales linearly with sequence length — > doubling context from 4k to 8k doubles the KV memory. > This is why long-context serving is so GPU-memory constrained.


In [ ]:
import pandas as pd

def kv_cache_size_gb(n_layers, n_kv_heads, head_dim, seq_len, batch_size, dtype_bytes=2):
    return (2 * n_layers * n_kv_heads * head_dim * seq_len * batch_size * dtype_bytes) / 1e9

model_configs = {
    "Qwen2.5-1.5B": dict(n_layers=28,  n_kv_heads=2,  head_dim=64),
    "Llama-3-8B":   dict(n_layers=32,  n_kv_heads=8,  head_dim=128),
    "Llama-3-70B":  dict(n_layers=80,  n_kv_heads=8,  head_dim=128),
    "GPT-4 (est.)":  dict(n_layers=120, n_kv_heads=16, head_dim=128),
}

seq_lengths = [512, 2048, 8192, 32768, 131072]
batch_size  = 8

rows = []
for label, cfg in model_configs.items():
    row = {"Model": label}
    for sl in seq_lengths:
        gb = kv_cache_size_gb(**cfg, seq_len=sl, batch_size=batch_size)
        key = f"seq={sl//1024}k" if sl >= 1024 else f"seq={sl}"
        row[key] = f"{gb:.2f} GB"
    rows.append(row)

df_kv = pd.DataFrame(rows).set_index("Model")
print(f"KV Cache Size (batch={batch_size}, BF16):")
try:
    from IPython.display import display; display(df_kv)
except Exception:
    print(df_kv.to_string())
print("\nFP8 KV: divide by 2  |  INT4 KV: divide by 4")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

seq_lens = np.array([512, 1024, 2048, 4096, 8192, 16384, 32768, 65536])
cfg = model_configs["Llama-3-8B"]
kv_bf16 = np.array([kv_cache_size_gb(**cfg, seq_len=s, batch_size=1, dtype_bytes=2) for s in seq_lens])

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(seq_lens/1024, kv_bf16,    "b-o", label="BF16 (2 bytes)")
ax.plot(seq_lens/1024, kv_bf16/2,  "g-s", label="FP8  (1 byte)")
ax.plot(seq_lens/1024, kv_bf16/4,  "r-^", label="INT4 (0.5 byte)")
ax.axhline(80, color="gray", linestyle="--", alpha=0.7, label="H100 80 GB limit")
ax.set_xlabel("Sequence Length (k tokens)")
ax.set_ylabel("KV Cache Size (GB)")
ax.set_title("Llama-3-8B: KV Cache Memory vs Sequence Length (batch=1)")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("assets/kv_cache_plot.png", dpi=120)
plt.show()

## 2.2 FP8 KV Cache Quantization

`register_hf_attentions_on_the_fly(model)` scans the model's HuggingFace attention classes and
patches them to insert KV quantizers on the K and V projections.
Unlike weight quantizers (calibrated offline), KV quantizers use **online** scale computation:
the scale is derived from the actual K/V tensor at each inference step.

In [ ]:
from modelopt.torch.quantization.plugins.huggingface import register_hf_attentions_on_the_fly

del model_export
free_memory()

model_kv = AutoModelForCausalLM.from_pretrained(
    model_name, torch_dtype=torch.bfloat16, device_map="cuda", trust_remote_code=True
).eval()

# Register HF attention class for KV cache quantization
register_hf_attentions_on_the_fly(model_kv)
print("KV attention registered.")

print("Applying FP8 weight + FP8 KV cache quantization ...")
model_fp8_kv = mtq.quantize(model_kv, mtq.FP8_DEFAULT_CFG, forward_loop=forward_loop)
fp8_kv_mem = gpu_mem_used()
print(f"FP8 + KV complete. GPU memory: {fp8_kv_mem:.0f} MB")

In [ ]:
model_bf16_ref = AutoModelForCausalLM.from_pretrained(
    model_name, torch_dtype=torch.bfloat16, device_map="cuda", trust_remote_code=True
).eval()

def peak_mem_at_seqlen(mdl, seq_len):
    ids = torch.randint(0, tokenizer.vocab_size, (1, seq_len), device="cuda")
    torch.cuda.reset_peak_memory_stats()
    with torch.no_grad(): mdl(input_ids=ids)
    return torch.cuda.max_memory_allocated() / 1e6

test_lens = [128, 256, 512, 1024, 2048]
mem_bf16_list, mem_fp8kv_list = [], []

print("Measuring peak memory vs sequence length ...")
for sl in test_lens:
    m_b = peak_mem_at_seqlen(model_bf16_ref, sl)
    m_f = peak_mem_at_seqlen(model_fp8_kv,  sl)
    mem_bf16_list.append(m_b); mem_fp8kv_list.append(m_f)
    print(f"  seq={sl:5d}: BF16={m_b:.0f} MB  FP8+KV={m_f:.0f} MB  reduction={m_b/m_f:.2f}x")

del model_bf16_ref, model_fp8_kv
free_memory()

## 2.3 Memory vs Sequence Length Plot

The plot below shows KV cache memory for BF16, FP8, and estimated INT4 across sequence lengths from 512 to 8,192 tokens, for the Qwen2.5-1.5B model loaded above.

**Reading the chart:**
- The gap between BF16 and FP8 lines is a constant 2× factor at every sequence length
- The INT4 line is an estimate (true INT4 KV requires a deployment backend);   shown here for reference
- The vertical line marks 4k tokens — a typical RAG / document QA sequence length

> **Key insight:** even for a 1.5B model, the KV cache at 8k tokens with batch size 32 > exceeds the weight memory. For 70B models at long context, the KV cache *dwarfs* the weights. > FP8 KV compression is arguably more impactful than weight quantization for serving workloads.


In [ ]:
import matplotlib.pyplot as plt

mem_int4_est = [m * 0.55 for m in mem_bf16_list]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ax = axes[0]
ax.plot(test_lens, mem_bf16_list,  "b-o", label="BF16 weights + KV")
ax.plot(test_lens, mem_fp8kv_list, "g-s", label="FP8 weights + FP8 KV")
ax.plot(test_lens, mem_int4_est,   "r-^", label="INT4 weights + KV (est.)")
ax.set_xlabel("Sequence Length"); ax.set_ylabel("Peak GPU Memory (MB)")
ax.set_title("Peak Memory vs Sequence Length"); ax.legend(); ax.grid(True, alpha=0.3)

ax2 = axes[1]
ax2.plot(test_lens, [b/f for b,f in zip(mem_bf16_list, mem_fp8kv_list)], "g-s", label="FP8 KV")
ax2.plot(test_lens, [b/f for b,f in zip(mem_bf16_list, mem_int4_est)],   "r-^", label="INT4 KV (est.)")
ax2.axhline(1.0, color="b", linestyle="--", alpha=0.5, label="BF16 baseline")
ax2.set_xlabel("Sequence Length"); ax2.set_ylabel("Reduction vs BF16")
ax2.set_title("Memory Reduction Factor"); ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("assets/kv_reduction_plot.png", dpi=120)
plt.show()

<h2 style="color: green;">✓ Module 2 — KV Cache Compression</h2>

**What you did:**
- Computed theoretical KV cache sizes for different models and sequence lengths
- Applied FP8 KV quantization via `register_hf_attentions_on_the_fly()`
- Verified the 2× memory reduction with a real memory measurement
- Plotted KV memory vs sequence length for BF16 / FP8 / INT4

**Key insight:** KV cache grows linearly with sequence length and batch size. At long context (8k+), it can exceed weight memory. FP8 KV compression is nearly lossless and should be enabled by default in production deployments.

→ Next module: **Cache Diffusion** — switch from LLMs to diffusion models.


---
# Module 3: Cache Diffusion (Denoising Step Caching)


## 3.0 Theory: Why Consecutive Steps Are Similar

In DDPM/DDIM diffusion, the denoising update is:

$$x_{t-1} = \frac{1}{\sqrt{\alpha_t}} \left(x_t - \frac{1-\alpha_t}{\sqrt{1-\bar{\alpha}_t}}
\epsilon_\theta(x_t, t)\right) + \sigma_t z$$

For consecutive steps $t$ and $t-1$, $x_t \approx x_{t-1}$ (they differ by a small noise update).
This means the hidden states and attention outputs of the UNet/DiT are also very similar across steps.

**DeepCache** [[arXiv:2312.00858](https://arxiv.org/abs/2312.00858)] and NVIDIA's Cache Diffusion
exploit this: skip the attention + FFN computation for most blocks at step $t$ and reuse outputs
from step $t-1$. Only selected blocks are recomputed every step.

```
Step t-1:  [Block A] compute -> store O_A
           [Block B] compute -> store O_B  <- always recompute (high-frequency)
           [Block C] compute -> store O_C

Step t:    [Block A] SKIP -> reuse O_A
           [Block B] compute -> new O_B
           [Block C] SKIP -> reuse O_C
```

This gives **~1.5–2.5× speedup** with minimal quality loss on 20-step schedules.

### The cachify API

```python
cachify.prepare(pipe, config_list)  # wrap blocks with CachedModule
cachify.enable(pipe)                # turn on caching
cachify.disable(pipe)               # turn off caching

with cachify.infer(pipe) as cached_pipe:  # auto-resets step counter
    img = cached_pipe(...)
```

The `config_list` specifies which blocks to cache and when:
```python
{
    "wildcard_or_filter_func": lambda name: "up_blocks.2" not in name,
    "select_cache_step_func":  lambda step: (step % 2) != 0,
}
```

Supported architectures: `UNet2DConditionModel` (SDXL), `PixArtTransformer2DModel`,
`SD3Transformer2DModel`, `UNetSpatioTemporalConditionModel` (SVD video).

> **Key Takeaway:** Cache diffusion is orthogonal to quantization — stack both for compounding speedups.
> It is training-free and works on any pretrained model without modification.

### 📚 Literature

- **DeepCache** — Ma et al., CVPR 2024. Training-free diffusion acceleration via high-level feature caching.  
  → [arXiv:2312.00858](https://arxiv.org/abs/2312.00858)
- **Learning-to-Cache (L2C)** — 2024. Learnable layer caching for diffusion transformers.  
  → [arXiv:2406.01733](https://arxiv.org/abs/2406.01733)


## 3.1 Load SDXL

We use **Stable Diffusion XL (SDXL)** as the primary diffusion model for this module. SDXL is a UNet-based text-to-image model with two text encoders (CLIP-L + CLIP-G) and a refinement stage. At 20 denoising steps, a single 1024×1024 image takes roughly 8–12 seconds on an A100.

The `DiffusionPipeline.from_pretrained()` call also adds the `cache_diffusion` module to the Python path — this is NVIDIA's implementation of denoising step caching, built on top of `diffusers`.

> ⏱️ **Expected load time:** ~60 s the first run (model deserialization from disk). > Subsequent loads in the same session are faster due to OS page cache.
>
> 📦 **Model:** `stabilityai/stable-diffusion-xl-base-1.0` > (~6.9 GB in BF16, loaded from the local HF cache)


In [ ]:
import sys
import torch
from diffusers import DiffusionPipeline

sys.path.insert(0, os.path.join(os.getcwd(), "Model-Optimizer/examples/diffusers/cache_diffusion"))

free_memory()

print("Loading SDXL base pipeline ...")
pipe = DiffusionPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.bfloat16,
        use_safetensors=True,
).to("cuda")
pipe.set_progress_bar_config(disable=True)

print(f"SDXL loaded. GPU memory: {gpu_mem_used():.0f} MB")
print(f"UNet class: {pipe.unet.__class__.__name__}")

## 3.2 Baseline Benchmark

We time SDXL at 20 denoising steps, averaging over 3 runs with a fixed seed. The `benchmark_pipe()` helper records:
- **Wall time** per image (seconds)
- **Peak GPU memory** during generation (MB)

This is the reference point every subsequent cache experiment compares against.

> ⏱️ **Expected runtime:** ~8–12 s / image on A100, ~4–6 s on H100. > Memory usage is dominated by activations during the UNet forward pass (~18 GB peak).


In [ ]:
import time

def benchmark_pipe(pipe, prompt, steps=20, n_runs=3, seed=42):
    """Returns (last_image, mean_latency_seconds)."""
    times, img = [], None
    for _ in range(n_runs):
        g = torch.Generator("cuda").manual_seed(seed)
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        img = pipe(prompt, num_inference_steps=steps, generator=g).images[0]
        torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)
    return img, sum(times) / len(times)

prompt = "a photorealistic mountain landscape at sunset, 4k, highly detailed"

print("Running SDXL baseline (20 steps, 3 runs) ...")
baseline_img, baseline_time = benchmark_pipe(pipe, prompt, steps=20, n_runs=3)
print(f"Baseline: {baseline_time:.2f}s  ({20/baseline_time:.1f} steps/sec)")

diffusion_results = {}
diffusion_results["SDXL BF16 (baseline)"] = {
    "model": "SDXL", "steps": 20, "latency_s": baseline_time, "speedup": 1.0
}
print(f"Stored SDXL baseline in diffusion_results")

## 3.3 Apply Cache Diffusion

`cachify.prepare(pipe, config_list)` wraps selected UNet blocks with `CachedModule`. The default SDXL config (`SDXL_DEFAULT_CONFIG`) caches the mid-block and most of the up-blocks, recomputing only the cross-attention blocks every step.

The two key parameters in the config:
- **`wildcard_or_filter_func`** — which blocks to cache (by module name)
- **`select_cache_step_func`** — at which denoising steps to use the cached output

```python
# SDXL_DEFAULT_CONFIG (simplified):
[{
    "wildcard_or_filter_func": lambda name: "up_blocks.2" not in name,
    "select_cache_step_func":  lambda step: (step % 2) != 0,
}]
```

This means: at every other step, skip all blocks except `up_blocks.2` (which handles fine-grained detail and must be recomputed every step to avoid blurring).

> 🔑 **Key line:** `cachify.enable(pipe)` — turns on caching for the next inference call. > Use `cachify.disable(pipe)` to restore standard inference.


In [ ]:
from cache_diffusion import cachify
from cache_diffusion.utils import SDXL_DEFAULT_CONFIG

print("Config: cache all blocks except up_blocks.2, on every odd step")
cachify.prepare(pipe, SDXL_DEFAULT_CONFIG)
cachify.enable(pipe)

print("Running cached benchmark (20 steps, 3 runs) ...")
with cachify.infer(pipe) as cached_pipe:
    cached_img, cached_time = benchmark_pipe(cached_pipe, prompt, steps=20, n_runs=3)

speedup = baseline_time / cached_time
print(f"Cache diffusion : {cached_time:.2f}s")
print(f"Speedup         : {speedup:.2f}x")

diffusion_results["SDXL + Cache Diffusion"] = {
    "model": "SDXL", "steps": 20, "latency_s": cached_time, "speedup": speedup
}
print(f"Stored SDXL+cache in diffusion_results")

## 3.4 Visual Comparison

Side-by-side: baseline (no caching) vs Cache Diffusion, same seed and prompt. Below the images we print the speedup factor and any visible quality difference.

> **What to look for:** the cached image should be nearly indistinguishable from the baseline > at a casual glance. Fine details (texture, small objects) may show minor smoothing, > but global composition and color are preserved — this is the expected trade-off.
>
> If speedup < 1.3×, the GPU is too fast relative to the cache overhead (uncommon on A100). > Typical speedup on A100: **~1.8–2.4×**.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(np.array(baseline_img))
axes[0].set_title(f"Baseline 20 steps\n{baseline_time:.2f}s"); axes[0].axis("off")
axes[1].imshow(np.array(cached_img))
axes[1].set_title(f"Cache Diffusion 20 steps\n{cached_time:.2f}s  ({speedup:.2f}x)"); axes[1].axis("off")
plt.suptitle("SDXL: same seed, same steps", fontsize=14)
plt.tight_layout()
plt.savefig("assets/cache_diffusion_compare.png", dpi=120, bbox_inches="tight")
plt.show()

## 3.5 Cache Diffusion vs Naive Step Reduction

Fewer denoising steps = lower quality. This experiment shows cache diffusion (20 steps)
beats naive step reduction (equivalent wall-time steps) in visual quality.

In [ ]:
cachify.disable(pipe)

equiv_steps = max(4, int(round(20 / speedup)))
print(f"Equivalent step count for same wall time: ~{equiv_steps} steps")

few_step_img, few_step_time = benchmark_pipe(pipe, prompt, steps=equiv_steps, n_runs=1)

cachify.enable(pipe)
with cachify.infer(pipe):
    cached20_img, _ = benchmark_pipe(pipe, prompt, steps=20, n_runs=1)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, img, title in zip(axes,
    [baseline_img, few_step_img, cached20_img],
    [f"20 steps (no cache)\n{baseline_time:.2f}s",
     f"{equiv_steps} steps (no cache)\n{few_step_time:.2f}s",
     f"20 steps + cache diffusion\n{cached_time:.2f}s ({speedup:.2f}x)"]):
    ax.imshow(np.array(img)); ax.set_title(title, fontsize=11); ax.axis("off")
plt.suptitle("Cache Diffusion vs Fewer Steps", fontsize=13)
plt.tight_layout()
plt.savefig("assets/cache_vs_fewer_steps.png", dpi=120, bbox_inches="tight")
plt.show()

## 3.6 PixArt-Alpha Demo

**PixArt-Alpha** is a Diffusion Transformer (DiT) — structurally different from SDXL's UNet. While SDXL processes spatial features through convolutional up/down-sampling blocks, PixArt applies transformer blocks uniformly across the image patches at a fixed resolution.

The point of this demo: `cachify` works on both UNet and DiT architectures through the same API. The only difference is the `config_list`, which references `PixArtTransformer2DModel` block names instead of UNet block names (`PIXART_DEFAULT_CONFIG` handles this automatically).

This architecture-agnostic design means the same caching strategy will apply to **NextDiT** (used in Z-Image), **SD3**, and other transformer-based diffusion models.

> 📦 **Model:** `PixArt-alpha/PixArt-XL-2-512x512` (~2.7 GB in BF16) > — smaller than SDXL, so generation is faster.


In [ ]:
from diffusers import PixArtAlphaPipeline
from cache_diffusion.utils import PIXART_DEFAULT_CONFIG

del pipe
free_memory()

print("Loading PixArt-Alpha 512px ...")
pipe_pa = PixArtAlphaPipeline.from_pretrained(
    "PixArt-alpha/PixArt-XL-2-512x512",
    torch_dtype=torch.bfloat16,   # bfloat16 avoids apex fused RMS norm issues with fp16
    use_safetensors=True,
).to("cuda")
pipe_pa.set_progress_bar_config(disable=True)

pa_base_img, pa_base_time = benchmark_pipe(pipe_pa, prompt, steps=20, n_runs=3)
print(f"PixArt baseline: {pa_base_time:.2f}s")

cachify.prepare(pipe_pa, PIXART_DEFAULT_CONFIG)
cachify.enable(pipe_pa)
with cachify.infer(pipe_pa) as p:
    pa_cached_img, pa_cached_time = benchmark_pipe(p, prompt, steps=20, n_runs=3)

pa_speedup = pa_base_time / pa_cached_time
print(f"PixArt + cache: {pa_cached_time:.2f}s  |  Speedup: {pa_speedup:.2f}x")

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(np.array(pa_base_img));   axes[0].set_title(f"Baseline\n{pa_base_time:.2f}s");   axes[0].axis("off")
axes[1].imshow(np.array(pa_cached_img)); axes[1].set_title(f"Cached\n{pa_cached_time:.2f}s ({pa_speedup:.2f}x)"); axes[1].axis("off")
plt.tight_layout()
plt.savefig("assets/pixart_cache.png", dpi=120, bbox_inches="tight")
plt.show()

del pipe_pa
free_memory()

diffusion_results["PixArt-Alpha BF16 (baseline)"] = {
    "model": "PixArt-Alpha", "steps": 20, "latency_s": pa_base_time, "speedup": 1.0
}
diffusion_results["PixArt-Alpha + Cache Diffusion"] = {
    "model": "PixArt-Alpha", "steps": 20, "latency_s": pa_cached_time, "speedup": pa_speedup
}
print(f"Stored PixArt results in diffusion_results")

<h2 style="color: green;">✓ Module 3 — Cache Diffusion</h2>

**What you did:**
- Benchmarked SDXL at 20 denoising steps (baseline)
- Applied `cachify` to skip ~50% of UNet block computations across steps
- Compared cached vs baseline images side-by-side
- Showed that cache diffusion beats naive step reduction at the same wall time
- Demonstrated architecture-agnostic caching on PixArt-Alpha (DiT backbone)

**Typical speedups:**
- SDXL on A100: **~1.8–2.4×**
- PixArt-Alpha on A100: **~2.0–2.5×**
- Quality loss: visually negligible at 20-step schedules

→ Next module: **AutoQuantization** — find the best per-layer precision automatically.


---
# Module 4: AutoQuantization & Mixed Precision


## 4.0 Theory: Why Uniform Quantization Is Suboptimal

Not all transformer layers are equally sensitive to quantization:

- **Input/output embeddings:** tied to vocabulary, very sensitive
- **First/last transformer blocks:** highest activation variance
- **Attention projections (Q, K, V):** directly affect attention patterns — sensitive
- **Middle MLP blocks:** often most tolerant of aggressive quantization

Uniform quantization (all layers to INT4, or all to FP8) leaves accuracy on the table:
sensitive layers are over-quantized, robust layers are under-compressed.

**AutoQuant** assigns different formats per layer under a global bit budget.
This idea was pioneered by **HAQ** [[arXiv:1811.08886](https://arxiv.org/abs/1811.08886)],
which used hardware-aware reinforcement learning to search bit-widths per layer.
ModelOpt's `auto_quantize` solves a more tractable formulation via **sensitivity scoring + LP**:

$$\min_{q \in \mathcal{Q}^L} \sum_l \mathcal{L}_l(q_l) \quad \text{s.t.} \quad
\frac{1}{L}\sum_l \text{bits}(q_l) \leq B$$

where $\mathcal{L}_l$ is the per-layer sensitivity score and $B$ is the target average bit-width.

### Three-Phase Process

1. **Calibrate each candidate format** (`num_calib_steps` batches per format):  
   Run calibration for FP8 and INT4-AWQ independently → get scale factors for each.

2. **Score per-layer sensitivity** (`num_score_steps` batches):  
   For each layer $l$ and format $q$, compute $\mathcal{L}_l(q)$ via gradient magnitude
   (how much the loss changes when layer $l$ uses format $q$) or KL divergence from BF16 outputs.

3. **LP solver** (CBC solver via PuLP):  
   Solve the integer program to assign one format per layer while meeting the `effective_bits` constraint.

**Why attention layers stay FP8 while FFN layers go INT4:**  
Attention projections (Q, K, V) have higher sensitivity scores — small perturbations in attention
weights propagate through the softmax nonlinearity and amplify errors. FFN layers are closer to
linear in their effect on the loss, making them more tolerant.

> **Key Takeaway:** AutoQuant typically recovers 0.5–1.0 PPL points vs uniform INT4 at the same
> average bit-width, with zero additional training cost.

### 📚 Literature

- **HAQ** — Wang et al., CVPR 2019 *(Oral)*. Hardware-aware automated mixed-precision quantization.  
  → [arXiv:1811.08886](https://arxiv.org/abs/1811.08886)
- **ZeroQuant-V2** — Yao et al., AAAI 2024. Comprehensive study of PTQ methods for LLMs.  
  → [arXiv:2303.08302](https://arxiv.org/abs/2303.08302)


## 4.1 AutoQuantize

`mtq.auto_quantize()` runs the three-phase process described in the theory section in a single call:

```python
model_autoq, search_results = mtq.auto_quantize(
    model,
    data_loader=calib_loader,
    loss_func=forward_loop_loss,
    constraints={"effective_bits": 6.0},
    quantization_formats=[mtq.FP8_DEFAULT_CFG, mtq.INT4_AWQ_CFG],
    num_calib_steps=64,
    num_score_steps=32,
    verbose=True,
)
```

**Parameters to understand:**
- **`effective_bits`** — target average bit-width across all quantized layers (here: 6 bits)
- **`quantization_formats`** — the candidate set; the LP solver picks one per layer
- **`num_calib_steps`** — calibration batches per format (more = better scale estimates)
- **`num_score_steps`** — sensitivity scoring batches (more = better layer ranking)

> ⏱️ **Expected runtime:** ~5–10 min on A100 for 1.5B model with default settings. > The verbose output prints phase transitions: `[Calibrating FP8]`, `[Calibrating INT4]`, > `[Scoring layers]`, `[Running LP solver]`.


In [ ]:
from modelopt.torch.utils.dataset_utils import get_dataset_dataloader
import modelopt.torch.quantization as mtq

free_memory()
model_aq = AutoModelForCausalLM.from_pretrained(
    model_name, torch_dtype=torch.bfloat16, device_map="cuda", trust_remote_code=True
).eval()

# Need labels in the batch to compute loss for sensitivity scoring
calib_loader = get_dataset_dataloader(
    dataset_name="cnn_dailymail", tokenizer=tokenizer,
    batch_size=2, num_samples=32, device="cuda",
    include_labels=True,
)

print("Running AutoQuantize (effective_bits=6.0) ...")
model_autoq, search_result = mtq.auto_quantize(
    model_aq,
    constraints={"effective_bits": 6.0},
    data_loader=calib_loader,
    forward_step=lambda m, b: m(**b),
    loss_func=lambda out, batch: out.loss,
    quantization_formats=[mtq.FP8_DEFAULT_CFG, mtq.INT4_AWQ_CFG],
    num_calib_steps=len(calib_loader),
    num_score_steps=len(calib_loader),
    verbose=True,
    disabled_layers=["*lm_head*"],
)

autoq_mem = gpu_mem_used()
print(f"\nAutoQuantize complete. GPU memory: {autoq_mem:.0f} MB")

## 4.2 Inspect Per-Layer Decisions

After `auto_quantize`, `search_results` contains the per-layer format assignments. The visualization below plots a colour-coded bar chart across all transformer layers:
- 🟦 **FP8** — sensitive layers (attention projections, first/last blocks)
- 🟧 **INT4** — robust layers (mid-stack MLP blocks)

**What to look for:**
- Layers 0 and the last 2–3 layers typically stay FP8 (high sensitivity)
- The QKV projections in every layer tend to stay FP8
- FFN `gate_proj` / `up_proj` / `down_proj` in middle layers go INT4

This distribution reflects the theory: attention weights amplify errors through softmax; FFN layers are more forgiving of coarser precision.

> **Try it:** change `effective_bits` from 6.0 to 5.0 and re-run. > More INT4 layers will appear — observe which layers "flip" first and why.


In [ ]:
from collections import defaultdict
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

format_counts = defaultdict(int)
layer_formats = {}

for name, module in model_autoq.named_modules():
    if hasattr(module, 'weight_quantizer') and module.weight_quantizer is not None:
        wq = module.weight_quantizer
        if hasattr(wq, 'num_bits'):
            fmt = "FP" + str(wq.num_bits) if getattr(wq, 'is_fp_quantizer', False) else "INT" + str(wq.num_bits)
        elif hasattr(wq, '_amax'):
            fmt = "FP8"
        else:
            fmt = "unknown"
        format_counts[fmt] += 1
        layer_formats[name] = fmt

print("Per-format layer counts:")
for fmt, cnt in sorted(format_counts.items()):
    print(f"  {fmt:10s}: {cnt} layers")

print("\nFirst 20 quantized layers:")
for name, fmt in list(layer_formats.items())[:20]:
    print(f"  {name[:60]:60s}  {fmt}")

color_map = {"FP8": "#2196F3", "INT4": "#F44336", "INT8": "#4CAF50", "unknown": "#9E9E9E"}
layer_names = list(layer_formats.keys())
colors = [color_map.get(layer_formats[n], "#9E9E9E") for n in layer_names]

fig, ax = plt.subplots(figsize=(14, 3))
for i, c in enumerate(colors):
    ax.bar(i, 1, color=c, width=0.9, edgecolor='none')
ax.set_xlim(-0.5, len(layer_names)-0.5); ax.set_yticks([])
ax.set_xlabel("Layer index")
ax.set_title(f"AutoQuant per-layer format ({model_name})")
patches = [mpatches.Patch(color=c, label=f) for f, c in color_map.items() if f in format_counts]
ax.legend(handles=patches)
plt.tight_layout()
plt.savefig("assets/autoquant_decisions.png", dpi=120, bbox_inches="tight")
plt.show()

## 4.3 Benchmark AutoQ vs Uniform Quantization

We compare four configurations head-to-head:

| Config | Expected avg bits | Expected memory | Expected quality |
|---|---|---|---|
| BF16 baseline | 16 | 100% | reference |
| Uniform INT4-AWQ | 4.0 | ~25% | acceptable |
| Uniform FP8 | 8.0 | ~50% | near-lossless |
| AutoQuant (6-bit target) | ~6.0 | ~37% | **best at this budget** |

The benchmark runs the same generation task for all four and records latency and GPU memory. Because we are in **simulation mode** on A100, latency comparisons are not meaningful — focus on the **memory numbers** and the **generated text quality**.

> **Expected result:** AutoQuant at 6-bit should produce output closer to BF16 than > uniform INT4, while using roughly half the memory of BF16. > This is the core value proposition of mixed-precision search.


In [ ]:
inputs_aq = tokenizer(prompt, return_tensors="pt").to("cuda")

def run_autoq():
    with torch.no_grad():
        return model_autoq.generate(
            **inputs_aq, max_new_tokens=100, do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

autoq_ms, autoq_std = benchmark_inference(run_autoq, n=5, warmup=2)
autoq_tokens_sec = 100 / (autoq_ms / 1000)

results["AutoQ (FP8+INT4, 6-bit avg)"] = {"mem_mb": autoq_mem, "latency_ms": autoq_ms, "tokens_sec": autoq_tokens_sec}

print(f"AutoQ (6-bit avg):")
print(f"  Latency    : {autoq_ms:.1f} ms")
print(f"  Throughput : {autoq_tokens_sec:.1f} tokens/sec")
print(f"  Memory     : {autoq_mem:.0f} MB")
print(f"  Speedup    : {baseline_ms/autoq_ms:.2f}x vs BF16")

# Qualitative generation test
with torch.no_grad():
    out = model_autoq.generate(**inputs_aq, max_new_tokens=150, do_sample=False,
                               pad_token_id=tokenizer.eos_token_id)
text = tokenizer.decode(out[0][inputs_aq["input_ids"].shape[-1]:], skip_special_tokens=True)
print(f"\nGeneration (AutoQ):\n{text[:400]}")

In [ ]:
import pandas as pd

bl_mem = results["BF16 (baseline)"]["mem_mb"]
bl_lat = results["BF16 (baseline)"]["latency_ms"]
rows = []
for name, r in results.items():
    rows.append({"Format": name,
                 "Mem (MB)": f"{r['mem_mb']:.0f}",
                 "Mem Reduction": f"{bl_mem/r['mem_mb']:.2f}x",
                 "Latency (ms)": f"{r['latency_ms']:.1f}",
                 "Speedup": f"{bl_lat/r['latency_ms']:.2f}x",
                 "Tokens/sec": f"{r['tokens_sec']:.1f}"})
df_m4 = pd.DataFrame(rows).set_index("Format")
try:
    from IPython.display import display; display(df_m4)
except Exception:
    print(df_m4.to_string())

<h2 style="color: green;">✓ Module 4 Complete — AutoQuantization & Mixed Precision</h2>

**What you did:**
- Ran `mtq.auto_quantize()` to let the LP solver assign FP8/INT4 per layer
- Inspected the per-layer decision map — attention layers stay FP8, mid-stack FFN goes INT4
- Benchmarked AutoQuant (6-bit target) against uniform INT4 and FP8

**Why it matters:** mixed-precision search consistently beats uniform quantization at the same average bit-width. The ~0.5–1 PPL improvement is free — it comes from redirecting the bit budget toward the layers that need it most.

→ Final module: **Combined Pipeline & Z-Image Outlook** — put it all together.


---
# Module 5: Combined Pipeline & Z-Image Outlook


## 5.0 The Optimization Landscape

The five techniques in this course are not alternatives — they target different bottlenecks
and can be stacked for **multiplicative** throughput improvements:

![The top five most impactful model optimization techniques](https://developer-blogs.nvidia.com/wp-content/uploads/2025/12/top-model-optimization-techniques-graphic-png.webp)

| Technique | Primary Bottleneck Addressed | Training Required? | Hardware Dependency |
|---|---|:-:|---|
| **PTQ** | Weight memory + GEMM compute | No | H100+ for FP8 native |
| **QAT** | Accuracy recovery at low precision | Yes | Any |
| **QAD** (QAT + Distillation) | Max accuracy + max compression | Yes | Any |
| **Speculative Decoding** | Autoregressive decode latency | No (needs draft model) | Any |
| **Pruning + Distillation** | Model size (permanent reduction) | Yes | Any |

### Decision Flowchart

```
Need faster/smaller model?
        │
        ▼
  Apply PTQ (FP8 or INT4-AWQ)
        │
        ├─ Accuracy OK? ──────────────────────────► Deploy ✓
        │
        ├─ Accuracy dropped?
        │       │
        │       ▼
        │   Apply QAT (fine-tune with quantization simulation)
        │       │
        │       ├─ Accuracy OK? ──────────────────► Deploy ✓
        │       │
        │       └─ Still degraded or need max quality?
        │               │
        │               ▼
        │           Apply QAD (QAT + Knowledge Distillation)
        │                                          ► Deploy ✓
        │
        └─ Model still too large for hardware?
                │
                ▼
            Pruning + Distillation (permanent size reduction)
                                               ► Deploy ✓
```

### 📚 Literature

- **Speculative Decoding** — Leviathan et al., ICML 2023. 2–3× latency reduction without weight changes.  
  → [arXiv:2211.17192](https://arxiv.org/abs/2211.17192)
- **Knowledge Distillation** — Hinton et al., 2015. Foundational paper on teacher–student training.  
  → [arXiv:1503.02531](https://arxiv.org/abs/1503.02531)


## 5.1 Stacking Optimizations

The techniques covered in this course are composable — each attacks a different bottleneck,
so their benefits multiply rather than overlap:

| Technique | Reduces | When it helps |
|---|---|---|
| FP8/INT4 weight quant | Weight memory + GEMM latency | Always (H100+) |
| FP8 KV cache | KV memory + bandwidth | Long sequences, large batch |
| Flash Attention | Attention memory $O(n)$ vs $O(n^2)$ | Sequence > 1k |
| Cache Diffusion | Total denoising compute | Diffusion models |
| AutoQuant | Accuracy vs compression tradeoff | When uniform quant degrades |

**Combined LLM serving stack** (FP8 weights + FP8 KV + Flash Attention):
- ~2.5–3× memory reduction
- ~2–3× throughput increase on H100
- <0.5% accuracy degradation

**Combined diffusion serving stack** (BF16/FP8 weights + Cache Diffusion):
- ~2× model memory reduction (with FP8)
- ~2–3× end-to-end speedup from caching
- Visually indistinguishable output at 20 steps

Knowledge distillation aligns the student's output probability distribution
with the teacher's — even after aggressive quantization or pruning:

![Knowledge distillation-trained student and teacher model outputs](https://developer-blogs.nvidia.com/wp-content/uploads/2025/12/knowledge-distillation-trained-student-teacher-model-outputs-png.webp)

*After distillation training, the quantized/pruned student model's output distribution
closely mirrors the full-precision teacher, recovering accuracy lost during compression.*


In [ ]:
from modelopt.torch.quantization.plugins.huggingface import register_hf_attentions_on_the_fly
import modelopt.torch.quantization as mtq
from modelopt.torch.utils.dataset_utils import create_forward_loop, get_dataset_dataloader

del model_autoq
free_memory()

model_combined = AutoModelForCausalLM.from_pretrained(
    model_name, torch_dtype=torch.bfloat16, device_map="cuda", trust_remote_code=True,
).eval()

attn_impl = getattr(model_combined.config, '_attn_implementation', 'default')
print(f"Attention implementation: {attn_impl}")

# Build fresh dataloader (previous one may be exhausted)
fwd_loop = create_forward_loop(dataloader=get_dataset_dataloader(
    dataset_name="cnn_dailymail", tokenizer=tokenizer,
    batch_size=4, num_samples=64, device="cuda",
))

# Stack: FP8 weights + FP8 KV
register_hf_attentions_on_the_fly(model_combined)
print("Applying FP8 weights + FP8 KV cache ...")
model_combined = mtq.quantize(model_combined, mtq.FP8_DEFAULT_CFG, forward_loop=fwd_loop)
mem_combined = gpu_mem_used()

inputs_c = tokenizer(prompt, return_tensors="pt").to("cuda")
def run_combined():
    with torch.no_grad():
        return model_combined.generate(
            **inputs_c, max_new_tokens=100, do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

combined_ms, combined_std = benchmark_inference(run_combined, n=5, warmup=2)
combined_toks = 100 / (combined_ms / 1000)

print(f"\nCombined (FP8 weights + FP8 KV):")
print(f"  Latency       : {combined_ms:.1f} ms")
print(f"  Throughput    : {combined_toks:.1f} tokens/sec")
print(f"  Memory        : {mem_combined:.0f} MB")
print(f"  Speedup       : {baseline_ms/combined_ms:.2f}x vs BF16")
print(f"  Mem reduction : {results['BF16 (baseline)']['mem_mb']/mem_combined:.2f}x vs BF16")

results["FP8 weights + FP8 KV"] = {
    "mem_mb": mem_combined, "latency_ms": combined_ms, "tokens_sec": combined_toks
}

## 5.2 Diffusion Model Quantization (Script-Based)

For large diffusion models (Flux-dev, SD3, WAN-14B), quantization is run as a standalone script rather than interactively — calibration alone takes 10–60 minutes.

The script `quantization/quantize.py` in the Model Optimizer repository handles the full pipeline: load model → calibrate → quantize → save checkpoint.

**Example: quantize Flux-schnell to FP8**
```bash
cd Model-Optimizer/examples/diffusers

python quantization/quantize.py \
    --model flux-schnell --model-dtype Float \
    --format fp8 --batch-size 1 --calib-size 256 \
    --quantize-mha \
    --n-steps 20 \
    --quantized-torch-ckpt-save-path ./flux-schnell-fp8.pt \
    --collect-method default
```

**Key flags:**
- `--format fp8` — target format (also supports `int4_awq`, `int8_sq`)
- `--quantize-mha` — also quantize multi-head attention KV cache to FP8
- `--calib-size 256` — number of prompts for calibration (256 is sufficient for diffusion)
- `--collect-method default` — scale collection method (min-max)

The cell below runs a **dry-run** (`--calib-size 1`, 1 step) to verify the script loads correctly without waiting for full calibration.


In [ ]:
cmd = [
    "python",
    os.path.join(os.getcwd(), "Model-Optimizer/examples/diffusers/quantization/quantize.py"),
    "--model", "flux-schnell",
    "--model-dtype", "Float",
    "--format", "fp8",
    "--batch-size", "1",
    "--calib-size", "16",
    "--n-steps", "4",
    "--quantize-mha",
    "--collect-method", "default",
    "--quantized-torch-ckpt-save-path", "./flux-schnell-fp8-demo.pt",
]
print("Command (run interactively after class):")
print(" ".join(cmd))
print()
print("(Skipped — full calib-size=256 takes ~10 min on H100)")
print()
print("Supported models:")
for m in ["sdxl-1.0", "sdxl-turbo", "sd3-medium", "sd3.5-medium",
           "flux-dev", "flux-schnell", "ltx-video-dev", "ltx-2",
           "wan2.2-t2v-14b", "wan2.2-t2v-5b"]:
    print(f"  --model {m}")

## 5.3 Z-Image Optimization Outlook

**Z-Image** is a new diffusion model built on the **NextDiT (Lumina2)** transformer backbone.

| Variant | Architecture | VAE? | Notes |
|---|---|---|---|
| `ZImage` | NextDiT (dim=3840) + Qwen3-4B | Yes | Latent-space DiT |
| `ZImagePixelSpace` | NextDiT pixel-space (DCT) | No | Simpler target |

### Quantization Targets

1. **NextDiT backbone (dim=3840)** — heaviest component. Architecturally similar to Flux DiT.
2. **Qwen3-4B text encoder** — already supports `quantization_metadata` via ComfyUI.
3. **VAE** (ZImage latent-space only) — INT8 calibration.
4. **ZImagePixelSpace**: no VAE, simpler optimization target.

### Adding Z-Image to models_utils.py

```python
class ModelType(str, Enum):
    ZIMAGE       = "zimage"
    ZIMAGE_PIXEL = "zimage-pixel"

_ZIMAGE_BASE_CONFIG: dict[str, Any] = {
    "backbone": "transformer",
    "dataset": _SD_PROMPTS_DATASET,
    "inference_extra_args": {
        "height": 512, "width": 512,
        "guidance_scale": 4.0, "max_sequence_length": 512,
    },
}

_ZIMAGE_PIXEL_CONFIG: dict[str, Any] = {
    "backbone": "transformer",  # no VAE
    "dataset": _SD_PROMPTS_DATASET,
    "inference_extra_args": {"height": 512, "width": 512, "guidance_scale": 4.0},
}

MODEL_REGISTRY[ModelType.ZIMAGE]       = "<org/ZImage>"
MODEL_REGISTRY[ModelType.ZIMAGE_PIXEL] = "<org/ZImage-PixelSpace>"
MODEL_DEFAULTS[ModelType.ZIMAGE]       = _ZIMAGE_BASE_CONFIG
MODEL_DEFAULTS[ModelType.ZIMAGE_PIXEL] = _ZIMAGE_PIXEL_CONFIG
```

### KV Cache for NextDiT

NextDiT has configurable `n_kv_heads` (GQA), already reducing KV memory by $H_q/H_{kv}$.
Stacking FP8 KV quantization gives another 2x.

### Cache Diffusion for NextDiT

Add NextDiT to the `CACHED_PIPE` registry in `cachify.py`:

```python
from comfy.ldm.lumina.model import NextDiT, NextDiTBlock
CACHED_PIPE[ZImagePipeline] = (NextDiTBlock,)
```

> **Research Opportunity:** `ZImagePixelSpace` operates in DCT-compressed pixel space.
> Its denoising dynamics may differ from latent-space models — optimal cache step selection
> patterns are an open research question.

## 5.4 Course Summary

### All Techniques

| Technique | Memory Reduction | Latency Reduction | Accuracy Impact | When to Use |
|---|---|---|---|---|
| FP8 PTQ (weights) | ~2x | 1.5-2x | <0.5% | Always on H100+ |
| INT8 SmoothQuant | ~2x | 1.3-1.8x | <1% | H100 or older GPUs |
| INT4-AWQ | ~4x | 1.5-2.5x | 1-2% | Memory-constrained |
| AutoQuant (mixed) | 2-4x | 1.5-2.5x | Better than uniform INT4 | Best accuracy/compression tradeoff |
| FP8 KV Cache | 2x KV | Up to 2x | <0.1% | Long context, large batch |
| Cache Diffusion | None | 1.5-2.5x | Visually negligible | Diffusion inference |
| GQA (arch.) | $H_q/H_{kv}$x | Proportional | None | Choose model with GQA |
| Flash Attention | $O(n)$ | 2-4x (long seq.) | None | Always, free |

### Decision Tree

```
Accuracy critical?      -> FP8 (H100) or INT8 SmoothQuant (older GPU)
Memory-constrained?     -> INT4-AWQ
Sequence > 4k tokens?   -> Add FP8 KV cache
Diffusion model?        -> Add Cache Diffusion
Uniform quant degrades? -> AutoQuant with mixed precision budget
Deploy to production?   -> export_hf_checkpoint() -> vLLM or TRT-LLM
```

### Key Numbers

- **FP8 on H100:** 2x GEMM throughput — hardware-native, near-free
- **KV cache formula:** $2 \times L \times H_{kv} \times d \times S \times B \times \text{bytes}$
- **Cache diffusion:** skip ~50% of transformer compute, keep >95% image quality
- **Calibration size:** 64-512 samples is sufficient for most PTQ
- **AutoQuant at 6-bit:** ~0.5-1 PPL better than uniform INT4

---

**Further Reading:**
- SmoothQuant: https://arxiv.org/abs/2211.10438
- AWQ: https://arxiv.org/abs/2306.00978
- DeepCache / Cache Diffusion: https://arxiv.org/abs/2312.00858
- NVIDIA Model Optimizer: https://nvidia.github.io/Model-Optimizer/
- Flash Attention: https://arxiv.org/abs/2205.14135
- H2O KV eviction: https://arxiv.org/abs/2306.14048

In [ ]:
import pandas as pd

del model_combined
free_memory()

print("=" * 65)
print(" COURSE COMPLETE: Final Results Summary")
print("=" * 65)

bl_mem = results["BF16 (baseline)"]["mem_mb"]
bl_lat = results["BF16 (baseline)"]["latency_ms"]

# ─── Section A: LLM / Transformer quantization ──────────────────────────────
print("\n── A. LLM Quantization (Qwen2.5-1.5B, 100 tokens) ─────────────")
llm_rows = []
for name, r in results.items():
    llm_rows.append({
        "Method": name,
        "GPU Mem (MB)": f"{r['mem_mb']:.0f}",
        "Mem Reduction": f"{bl_mem/r['mem_mb']:.2f}x",
        "Latency (ms)": f"{r['latency_ms']:.1f}",
        "Speedup": f"{bl_lat/r['latency_ms']:.2f}x",
        "Tokens/sec": f"{r['tokens_sec']:.1f}",
    })
df_llm = pd.DataFrame(llm_rows).set_index("Method")
try:
    from IPython.display import display; display(df_llm)
except Exception:
    print(df_llm.to_string())

# ─── Section B: Diffusion model step caching ────────────────────────────────
print("\n── B. Cache Diffusion (20 denoising steps) ──────────────────────")
if "diffusion_results" in dir():
    diff_rows = []
    for name, r in diffusion_results.items():
        diff_rows.append({
            "Method": name,
            "Model": r["model"],
            "Steps": r["steps"],
            "Latency (s)": f"{r['latency_s']:.2f}",
            "Speedup": f"{r['speedup']:.2f}x",
            "Steps/sec": f"{r['steps']/r['latency_s']:.2f}",
        })
    df_diff = pd.DataFrame(diff_rows).set_index("Method")
    try:
        from IPython.display import display; display(df_diff)
    except Exception:
        print(df_diff.to_string())
else:
    print("  (diffusion_results not available — run Module 3 cells first)")

# ─── Combined one-pager ──────────────────────────────────────────────────────
print("\n── All results (one-pager) ──────────────────────────────────────")
all_rows = []
for name, r in results.items():
    all_rows.append({
        "Method": name,
        "Domain": "LLM",
        "Metric": f"{r['latency_ms']:.0f} ms / {r['tokens_sec']:.0f} tok/s",
        "Memory": f"{r['mem_mb']:.0f} MB",
        "Speedup": f"{bl_lat/r['latency_ms']:.2f}x",
    })
if "diffusion_results" in dir():
    for name, r in diffusion_results.items():
        all_rows.append({
            "Method": name,
            "Domain": "Diffusion",
            "Metric": f"{r['latency_s']:.2f} s / 20 steps",
            "Memory": "—",
            "Speedup": f"{r['speedup']:.2f}x",
        })
df_all = pd.DataFrame(all_rows).set_index("Method")
try:
    from IPython.display import display; display(df_all)
except Exception:
    print(df_all.to_string())

print()
print("Saved artifacts in assets/:")
for f in ["kv_cache_plot.png", "kv_reduction_plot.png",
           "cache_diffusion_compare.png", "cache_vs_fewer_steps.png",
           "pixart_cache.png", "autoquant_decisions.png",
           "appendix_a_gemm_paths.png", "exports/qwen-fp8/"]:
    print(f"  {f}")


---

# Appendix A: Why PTQ Speedup Requires H100+ and a Deployment Backend

> **Context:** If you ran this notebook on a DGX Spark (NVIDIA GB10) or an older GPU, you likely
> observed that FP8/INT4 quantization made inference *slower*, not faster. This appendix explains
> exactly why, and what changes on H100 and Blackwell architectures.

## A.1 The Two Execution Paths

There are two fundamentally different ways to run a quantized linear layer $Y = XW$:

### Path 1 — Simulation Mode (what ModelOpt does by default)

This is what runs when you call `mtq.quantize()` and immediately benchmark in PyTorch:

```
Input X  (BF16)
   │
   ▼
[TensorQuantizer]  ←── scale s computed from calibration
   │  x_q = clamp(round(X / s), -127, 127)   # quantize to INT8 / FP8
   │  x̂   = x_q * s                           # dequantize back to BF16
   ▼
Standard BF16 GEMM: Y = x̂ · W̃
   ▲
   │
[TensorQuantizer on W]
   │  W_q = clamp(round(W / s_w), -127, 127)  # quantize
   └─ W̃   = W_q * s_w                         # dequantize back to BF16
```

**Net result:** the GEMM itself still runs in **BF16**. The quantizers add two extra operations
(quantize + dequantize) on top. This is *more* work than baseline BF16 → **slower**.

The purpose of simulation mode is not speed — it is **calibration accuracy**: finding the scale
factors $s$ that minimise the quantization error $\|X - \hat{X}\|$, and **export**: saving the
quantized weights + scales to disk so a deployment backend can use them.

### Path 2 — Native Execution Mode (H100 / Blackwell + TRT-LLM / vLLM)

On H100, the same linear layer runs as:

```
Input X  (BF16)
   │
   ▼
[Cast to FP8]  — single element-wise op, ~0 overhead
   │
   ▼
FP8 Tensor Core GEMM: Y = X_fp8 · W_fp8
   │
   │   ← hardware executes this at 2x the TFLOPS of BF16 GEMM
   │   ← half the memory bandwidth consumed
   ▼
Output Y (BF16 or FP32 accumulation)
```

No dequantize step. The GEMM kernel itself accepts FP8 inputs natively.
The scale factors are fused into the kernel as scalar multipliers.

**Net result:** ~2x throughput, ~2x memory bandwidth reduction, near-zero overhead.

## A.2 What Makes H100 Different: Tensor Core Generations

NVIDIA has progressively added lower-precision Tensor Core support across GPU generations:

| GPU | Architecture | Native FP8 GEMM | Native INT8 GEMM | Native INT4 GEMM | Notes |
|-----|-------------|:-:|:-:|:-:|---|
| V100 | Volta | ✗ | ✗ | ✗ | FP16/FP32 only |
| A100 | Ampere | ✗ | ✓ | ✓ | INT8 2x over FP16; INT4 4x |
| H100 | Hopper | ✓ | ✓ | ✓ | FP8 2x over BF16; first-gen FP8 HW |
| H200 | Hopper | ✓ | ✓ | ✓ | Same compute, larger HBM3e |
| B100/B200 | Blackwell | ✓ (NVFP4) | ✓ | ✓ | FP4 Tensor Cores — 4x over BF16 |
| GB10 (DGX Spark) | Blackwell | ✓ | ✓ | ✓ | ARM+GPU SoC; PyTorch dispatch not yet optimised for FP8 GEMM via cuBLAS |

**The key insight:** native hardware support is necessary but not sufficient. The *software stack*
must also dispatch to the right kernel. PyTorch's `nn.Linear` does **not** call FP8 cuBLAS
kernels — it always uses the BF16/FP32 path. Only TRT-LLM, vLLM, and SGLang inject the
optimised kernels at inference time.

## A.3 The Roofline Model: Why Lower Precision Helps

A transformer GEMM for token generation is **memory-bandwidth bound**, not compute bound
(batch=1, sequence is short). The roofline model gives the achievable FLOP/s:

$$\text{Throughput} = \min\!\left(\text{Peak TFLOPS},\; \frac{\text{Memory BW}}{\text{Arithmetic Intensity}}\right)$$

For BF16 on H100:
- Peak BF16 TFLOPS ≈ 989 TF (with sparsity)
- Memory BW ≈ 3.35 TB/s
- Arithmetic intensity of a GEMM with batch=1 is very low (~1 FLOP/byte)
- → **memory-bandwidth bound**

For FP8:
- Weights are 2x smaller → 2x fewer bytes loaded per token
- → arithmetic intensity doubles → 2x throughput (still memory-bound, but higher ceiling)

This is why FP8 helps even for small-batch inference: it is not about TFLOPS,
it is about **halving the bytes read from HBM per generated token**.

## A.4 INT4-AWQ: A Special Case

INT4-AWQ is *always* in simulation mode in PyTorch, even on H100. The reason:

- H100 has no INT4 Tensor Core that accepts asymmetric group-quantised weights directly.
- INT4 gains come from **dequant-then-BF16-GEMM** with optimised dequant kernels (Marlin, GPTQ-cuda).
- These custom CUDA kernels are fused: dequant + GEMM in a single kernel, avoiding extra memory traffic.
- In vanilla PyTorch, the dequant and GEMM are separate ops → slow.

**Marlin kernel (used by vLLM for INT4-AWQ):**
```
W_int4  (stored, 4-bit)  →  [fused dequant + GEMM]  →  Y (BF16)
```
The dequant happens in register file, never materialising the BF16 weights in HBM.
Memory traffic ≈ 4 bits/param vs 16 bits/param → 4x bandwidth savings.


## A.5 The Full Workflow (Summary)

```
┌─────────────────────────────────────────────────────┐
│  CALIBRATION PHASE  (this notebook, any GPU)        │
│                                                     │
│  mtq.quantize(model, FP8_DEFAULT_CFG, forward_loop) │
│    └─ inserts TensorQuantizers                      │
│    └─ runs forward passes → collects amax stats     │
│    └─ freezes scale factors s per tensor            │
│                                                     │
│  export_hf_checkpoint(model, export_dir)            │
│    └─ serialises weights as FP8/INT4 + scale        │
└──────────────────────┬──────────────────────────────┘
                       │  checkpoint on disk
                       ▼
┌─────────────────────────────────────────────────────┐
│  DEPLOYMENT PHASE  (production server, H100/B100)   │
│                                                     │
│  vLLM:       vllm serve <export_dir>                │
│    └─ loads FP8 weights, dispatches to              │
│       cuBLAS FP8 GEMM or FlashInfer FP8 attention   │
│                                                     │
│  TRT-LLM:    trtllm-build + trtllm-run              │
│    └─ compiles ONNX graph → TRT engine              │
│    └─ all GEMMs replaced with FP8 trt kernels       │
│                                                     │
│  SGLang:     python -m sglang.launch_server         │
│    └─ same FP8 kernel path as vLLM                  │
└─────────────────────────────────────────────────────┘
```

> **Bottom line:** ModelOpt's job is to find the right scale factors accurately and save them.
> The deployment backend's job is to use those scales to run fast native kernels.
> Benchmarking in PyTorch only tests the first half of this pipeline.

In [ ]:
"""
Appendix A — Code: Measure the simulation overhead directly.

We time each step of Path 1 (simulation) on a representative GEMM size
and compare to a plain BF16 GEMM of the same dimensions.
"""
import torch
import time
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

device = "cuda"
torch.cuda.synchronize()

# ── dimensions representative of one Qwen2.5-1.5B MLP layer ──────────────
BATCH, SEQ, D_IN, D_OUT = 1, 128, 2048, 8192
dtype = torch.bfloat16

X = torch.randn(BATCH * SEQ, D_IN,  device=device, dtype=dtype)
W = torch.randn(D_OUT,       D_IN,  device=device, dtype=dtype)

def timed(fn, n=200, warmup=50):
    for _ in range(warmup): fn()
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(n): fn()
    torch.cuda.synchronize()
    return (time.perf_counter() - t0) / n * 1e6   # µs per call

# ── Step 0: plain BF16 GEMM (baseline) ───────────────────────────────────
t_gemm_bf16 = timed(lambda: torch.nn.functional.linear(X, W))
print(f"BF16 GEMM:                  {t_gemm_bf16:6.1f} µs")

# ── Step 1: simulate FP8 quantization (clamp + round + scale) ─────────────
amax_x = X.abs().max()
scale_x = amax_x / 448.0   # FP8 E4M3 max value ≈ 448

def quantize_fp8(t, scale):
    return t.div(scale).clamp(-448, 448).to(torch.float8_e4m3fn)

def dequantize_fp8(t_q, scale):
    return t_q.to(torch.bfloat16).mul(scale)

t_quant   = timed(lambda: quantize_fp8(X, scale_x))
t_dequant = timed(lambda: dequantize_fp8(quantize_fp8(X, scale_x), scale_x))
print(f"Quantize X to FP8:          {t_quant:6.1f} µs")
print(f"Quantize + Dequantize X:    {t_dequant:6.1f} µs")

# ── Step 2: full simulation path = quant(X) + dequant(X) + quant(W) + dequant(W) + GEMM ──
amax_w  = W.abs().max()
scale_w = amax_w / 448.0

def simulation_path():
    x_hat = dequantize_fp8(quantize_fp8(X, scale_x), scale_x)
    w_hat = dequantize_fp8(quantize_fp8(W, scale_w), scale_w)
    return torch.nn.functional.linear(x_hat, w_hat)

t_simulation = timed(simulation_path)
print(f"Full simulation path:        {t_simulation:6.1f} µs  "
      f"({t_simulation/t_gemm_bf16:.2f}x BF16 baseline)")

# ── Step 3: native FP8 GEMM via torch._scaled_mm (H100 only) ────────────
native_fp8_available = False
t_native_fp8 = None
try:
    X_fp8 = quantize_fp8(X, scale_x)
    W_fp8 = quantize_fp8(W, scale_w)
    # torch._scaled_mm requires 2D inputs and scale tensors
    scale_x_t = scale_x.reshape(1)
    scale_w_t = scale_w.reshape(1)
    def native_fp8_gemm():
        return torch._scaled_mm(
            X_fp8, W_fp8.t(),
            scale_a=scale_x_t, scale_b=scale_w_t,
            out_dtype=torch.bfloat16,
        )
    native_fp8_gemm()   # test run
    t_native_fp8 = timed(native_fp8_gemm)
    native_fp8_available = True
    print(f"Native FP8 GEMM (_scaled_mm): {t_native_fp8:6.1f} µs  "
          f"({t_gemm_bf16/t_native_fp8:.2f}x BF16 baseline)  ← H100+ only")
except Exception as e:
    print(f"Native FP8 GEMM: NOT available on this GPU ({type(e).__name__}: {e})")
    print("  → Expected on GB10/DGX Spark; would show ~2x speedup on H100.")

# ── Plot: breakdown bar chart ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: absolute timing breakdown
labels_sim  = ["BF16 GEMM\n(baseline)", "Quant X\n(overhead)", "Dequant X\n(overhead)",
                "Quant W\n(overhead)", "Dequant W\n(overhead)", "BF16 GEMM\n(in sim)"]
heights_sim = [t_gemm_bf16,
               t_quant, t_quant,          # approx same for W
               t_quant, t_quant,
               t_gemm_bf16]
colors_sim  = ["#2196F3", "#FF5722", "#FF5722", "#FF5722", "#FF5722", "#2196F3"]
bars = axes[0].bar(labels_sim, heights_sim, color=colors_sim, edgecolor="white", linewidth=0.5)
axes[0].set_ylabel("Time per call (µs)")
axes[0].set_title("Simulation mode: overhead breakdown\n(Path 1 — ModelOpt in PyTorch)")
axes[0].grid(axis="y", alpha=0.3)
blue  = mpatches.Patch(color="#2196F3", label="Useful work (GEMM)")
red   = mpatches.Patch(color="#FF5722", label="Simulation overhead")
axes[0].legend(handles=[blue, red])

# Right: total latency comparison across paths
path_labels  = ["BF16 GEMM\n(no quant)", "Simulation\n(Path 1)", "Native FP8\n(Path 2 / H100)"]
path_times   = [t_gemm_bf16, t_simulation, t_native_fp8 if native_fp8_available else t_gemm_bf16 * 0.5]
path_colors  = ["#2196F3", "#FF5722", "#4CAF50"]
path_hatches = ["", "", "" if native_fp8_available else "//"]
for i, (h, c, hatch) in enumerate(zip(path_times, path_colors, path_hatches)):
    axes[1].bar(path_labels[i], h, color=c, hatch=hatch, edgecolor="white")
if not native_fp8_available:
    axes[1].text(2, path_times[2] * 1.05, "estimated\n(~0.5x BF16)", ha="center",
                 fontsize=9, style="italic", color="#4CAF50")
axes[1].axhline(t_gemm_bf16, color="#2196F3", linestyle="--", alpha=0.5, label="BF16 baseline")
axes[1].set_ylabel("Time per GEMM call (µs)")
axes[1].set_title(f"Total latency by execution path\n(GEMM: {BATCH*SEQ}×{D_IN}×{D_OUT})")
axes[1].legend()
axes[1].grid(axis="y", alpha=0.3)

plt.suptitle("Appendix A: Simulation vs Native FP8 Execution", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("assets/appendix_a_gemm_paths.png", dpi=130, bbox_inches="tight")
plt.show()

print()
print("Summary:")
print(f"  BF16 GEMM baseline  : {t_gemm_bf16:.1f} µs  (1.00x)")
print(f"  Simulation (Path 1) : {t_simulation:.1f} µs  ({t_simulation/t_gemm_bf16:.2f}x — SLOWER)")
if native_fp8_available:
    print(f"  Native FP8 (Path 2) : {t_native_fp8:.1f} µs  ({t_gemm_bf16/t_native_fp8:.2f}x — FASTER)")
else:
    print(f"  Native FP8 (Path 2) : not dispatched on this GPU in PyTorch")
    print(f"                        → deploy via vLLM / TRT-LLM on H100 for ~2x speedup")

---

# Appendix B: Adding Z-Image Support to NVIDIA Model Optimizer

> **Goal:** Quantize Z-Image (6B S3-DiT) with FP8/INT8, apply FP8 KV cache quantization,
> and export a checkpoint usable in ComfyUI or vLLM-style serving.

## B.0 Z-Image Architecture Recap

Z-Image is built on the **S3-DiT (Single-Stream Diffusion Transformer)** architecture
from the paper *arXiv:2511.22699*. Unlike Flux (which has dual-stream + single-stream blocks),
Z-Image concatenates text tokens, visual semantic tokens, and image VAE tokens into
a **single unified sequence** from layer 1.

```
┌──────────────────────────────────────────────────────┐
│  Z-Image pipeline (ZImagePipeline, diffusers)        │
│                                                      │
│  ┌────────────────┐   ┌───────────────────────────┐  │
│  │  Text encoder  │   │  Transformer (backbone)   │  │
│  │  Qwen3-4B      │   │  NextDiT  dim=3840        │  │
│  │  (2560-dim)    │   │  S3-DiT: unified seq      │  │
│  └───────┬────────┘   │  JointAttention (GQA)     │  │
│          │ text_emb   │  SwiGLU FFN               │  │
│          └────────────►  48 layers                │  │
│                       └──────────────┬────────────┘  │
│  ┌────────────────┐                  │ latents        │
│  │  VAE           │◄─────────────────┘               │
│  │  (decode only) │   latent → pixel                 │
│  └────────────────┘                                  │
└──────────────────────────────────────────────────────┘
```

**Quantization targets and priority:**

| Component | Size | Priority | Technique |
|-----------|------|----------|-----------|
| NextDiT transformer | ~12 GB (BF16) | **High** | FP8 weights + FP8 KV cache |
| Qwen3-4B text encoder | ~8 GB (BF16) | Medium | FP8 or INT8 SmoothQuant |
| VAE decoder | ~0.3 GB | Low | INT8 (optional) |


## B.1 What Files Need to Change

The quantization script lives in:
```
Model-Optimizer/examples/diffusers/quantization/
├── models_utils.py   ← ADD: ModelType, MODEL_REGISTRY, MODEL_PIPELINE, MODEL_DEFAULTS
├── utils.py          ← ADD: filter_func_zimage()
└── quantize.py       ← no changes needed (generic CLI)
```

Three additions, then run the existing `quantize.py` CLI unchanged.

## B.2 Step 1 — Add the Filter Function (`utils.py`)

The filter function tells ModelOpt which layers to **skip** (return `True` = skip).
We skip layers that are:
- **Embedding layers** — patch embedder, positional embedder, timestep/caption embedder.
  These are small and their weight distributions are very non-uniform.
- **Final output projection** — `final_layer` maps hidden dim → patch dim.
  Quantizing the last layer degrades image quality disproportionately.
- **Normalization layers** — RMSNorm weights are tiny (1D), quantizing them adds
  overhead with no compression benefit.

```python
# In utils.py — add after filter_func_flux_dev()

import re

def filter_func_zimage(name: str) -> bool:
    """Filter function for Z-Image (NextDiT / S3-DiT backbone).

    Returns True for layers that should NOT be quantized.
    Skips: patch embedder, timestep/caption embedding, final projection, norms.
    Quantizes: all JointAttention (qkv, out) and FFN (w1, w2, w3) linears.
    """
    pattern = re.compile(
        r".*("
        r"x_embedder"           # patch embedding: image patches → dim
        r"|final_layer"         # output projection: dim → patch_size²×channels
        r"|time_caption_embed"  # timestep + caption conditioning MLP
        r"|cap_embedder"        # caption projection (if present)
        r"|norm"                # all RMSNorm / LayerNorm weight tensors
        r"|pos_embed"           # any fixed positional embeddings
        r").*"
    )
    return pattern.match(name) is not None
```

**Why these and not others?**

```
NextDiTBlock (×48)
├── JointAttention
│   ├── qkv  (Linear 3840 → 3×head_dim×n_heads)  ← QUANTIZE ✓
│   └── out  (Linear head_dim×n_heads → 3840)     ← QUANTIZE ✓
├── feed_forward
│   ├── w1   (Linear 3840 → ffn_dim, gate)        ← QUANTIZE ✓
│   ├── w2   (Linear ffn_dim → 3840)              ← QUANTIZE ✓
│   └── w3   (Linear 3840 → ffn_dim, value)       ← QUANTIZE ✓
├── norm1 / norm2  (RMSNorm — 1D weight)          ← SKIP ✗
└── adaLN_modulation (small MLP, 3840→2×3840)     ← optional: include
```

Together, `qkv + out + w1 + w2 + w3` account for ~95% of transformer parameter count.
Skipping the norms (<0.1% of params) costs nothing in compression.

## B.3 Step 2 — Register in `models_utils.py`

```python
# ── 1. Import ZImagePipeline ───────────────────────────────────────────────
from diffusers import ZImagePipeline   # available in diffusers >= 0.33

# ── 2. Extend the ModelType enum ──────────────────────────────────────────
class ModelType(str, Enum):
    # ... existing entries ...
    ZIMAGE       = "zimage"
    ZIMAGE_TURBO = "zimage-turbo"

# ── 3. Register HuggingFace model IDs ─────────────────────────────────────
MODEL_REGISTRY[ModelType.ZIMAGE]       = "Tongyi-MAI/Z-Image"
MODEL_REGISTRY[ModelType.ZIMAGE_TURBO] = "Tongyi-MAI/Z-Image-Turbo"

# ── 4. Register pipeline classes ──────────────────────────────────────────
MODEL_PIPELINE[ModelType.ZIMAGE]       = ZImagePipeline
MODEL_PIPELINE[ModelType.ZIMAGE_TURBO] = ZImagePipeline

# ── 5. Calibration defaults ───────────────────────────────────────────────
_ZIMAGE_BASE_CONFIG: dict[str, Any] = {
    "backbone": "transformer",          # pipe.transformer = NextDiT
    "dataset":  _SD_PROMPTS_DATASET,    # reuse Stable Diffusion prompts dataset
    "inference_extra_args": {
        "height": 512,                  # use 512×512 for fast calibration
        "width":  512,                  # (production: 1024×1024 or 1280×720)
        "num_inference_steps": 20,      # fewer steps = faster calibration
        "guidance_scale": 4.0,
        "cfg_normalization": False,     # False = stylistic mode (default)
    },
}

MODEL_DEFAULTS[ModelType.ZIMAGE]       = _ZIMAGE_BASE_CONFIG
MODEL_DEFAULTS[ModelType.ZIMAGE_TURBO] = {
    **_ZIMAGE_BASE_CONFIG,
    "inference_extra_args": {
        **_ZIMAGE_BASE_CONFIG["inference_extra_args"],
        "num_inference_steps": 8,       # Turbo uses 8 steps, no CFG
        "guidance_scale": 1.0,
    },
}

# ── 6. Wire up the filter function ────────────────────────────────────────
# In get_model_filter_func():
filter_func_map[ModelType.ZIMAGE]       = filter_func_zimage
filter_func_map[ModelType.ZIMAGE_TURBO] = filter_func_zimage
```

## B.4 Step 3 — Run Quantization

Once the two files are edited, run the existing CLI — no other changes needed:

```bash
cd Model-Optimizer/examples/diffusers

export NVTE_APEX_FUSED_LAYERNORM=0
export APEX_FUSED_RMSNORM=0

# FP8 quantization of the transformer backbone
python quantization/quantize.py \
    --model zimage \
    --model-dtype BFloat16 \
    --format fp8 \
    --batch-size 1 \
    --calib-size 128 \
    --n-steps 20 \
    --collect-method default \
    --quantize-mha \
    --quantized-torch-ckpt-save-path ./zimage-transformer-fp8.pt
```

**Flag notes:**
- `--backbone transformer` is the default (set in `MODEL_DEFAULTS`) — quantizes `pipe.transformer`.
  Pass `--backbone text_encoder` in a second run to quantize the Qwen3-4B encoder separately.
- `--quantize-mha` enables FP8 KV cache quantization on attention outputs (see B.5).
- `--calib-size 128` with 20 steps means 128 × 20 = 2560 forward passes through the transformer.
  Increase to 256 for better calibration; 64 is acceptable for a quick test.

**Expected runtime:** ~15–25 min on H100 SXM5 for 128 samples at 512×512.

## B.5 Step 4 — KV Cache Quantization (Manual API)

`--quantize-mha` in the CLI handles this automatically for supported attention types.
For manual control or custom pipelines, the API is:

```python
import torch
from diffusers import ZImagePipeline
from modelopt.torch.quantization.plugins.huggingface import register_hf_attentions_on_the_fly
import modelopt.torch.quantization as mtq

pipe = ZImagePipeline.from_pretrained(
    "Tongyi-MAI/Z-Image",
    torch_dtype=torch.bfloat16,
).to("cuda")

# Register Z-Image's JointAttention for KV quantization.
# register_hf_attentions_on_the_fly walks the model and patches
# every attention module it finds (works for any HF-style model).
register_hf_attentions_on_the_fly(pipe.transformer)

# Now quantize — KV quantizers are included automatically
pipe.transformer = mtq.quantize(pipe.transformer, mtq.FP8_DEFAULT_CFG, forward_loop)
```

**What this does to the attention computation:**

```
Before (BF16 KV):
  K, V = qkv(x).split(...)          # K,V stored as BF16  → 2 bytes/elem
  attn = softmax(Q @ K.T / √d) @ V

After (FP8 KV):
  K, V = qkv(x).split(...)
  K_q = quantize_fp8(K, scale_K)    # K stored as FP8    → 1 byte/elem  ← 2x smaller
  V_q = quantize_fp8(V, scale_V)    # V stored as FP8    → 1 byte/elem
  attn = softmax(Q @ dequant(K_q).T / √d) @ dequant(V_q)
```

**Memory saving for Z-Image at 1024×1024, batch=1:**

```python
# NextDiT: 48 layers, n_kv_heads=16, head_dim=240 (dim=3840, n_heads=16)
n_layers, n_kv_heads, head_dim = 48, 16, 240
seq_len = (1024 // 2) * (1024 // 2)   # VAE downscale 2x, patch size 2 → 512×512 = 262144 tokens
kv_bf16 = 2 * n_layers * n_kv_heads * head_dim * seq_len * 2 / 1e9
kv_fp8  = kv_bf16 / 2
print(f"KV cache BF16: {kv_bf16:.1f} GB")
print(f"KV cache FP8:  {kv_fp8:.1f} GB")
```

## B.6 Step 5 — Quantize the Text Encoder (Qwen3-4B) Separately

The Qwen3-4B encoder adds ~8 GB BF16. Since it runs once per prompt (not per denoising step),
it dominates memory but not latency. FP8 halves its footprint.

```bash
# Second pass: quantize only the text encoder
python quantization/quantize.py \
    --model zimage \
    --model-dtype BFloat16 \
    --backbone text_encoder \
    --format fp8 \
    --batch-size 1 \
    --calib-size 64 \
    --n-steps 1 \
    --quantized-torch-ckpt-save-path ./zimage-text-encoder-fp8.pt
```

Or manually with INT8 SmoothQuant (safer for text encoders with outlier activations):

```python
from transformers import AutoTokenizer

# Load text encoder sub-model
text_encoder = pipe.text_encoder.to("cuda")

# Build text calibration forward loop
sample_prompts = ["a beautiful landscape", "portrait photography", ...]
enc = AutoTokenizer.from_pretrained("Tongyi-MAI/Z-Image", subfolder="tokenizer")

def text_forward_loop(model):
    for prompt in sample_prompts:
        tokens = enc(prompt, return_tensors="pt").to("cuda")
        model(**tokens)

text_encoder = mtq.quantize(text_encoder, mtq.INT8_SMOOTHQUANT_CFG, text_forward_loop)
```

## B.7 Step 6 — Export and Load in ComfyUI

### Export with ModelOpt

```python
from modelopt.torch.export import export_hf_checkpoint

# Export transformer
export_hf_checkpoint(pipe.transformer, export_dir="./zimage-fp8/transformer/")

# Export text encoder (if quantized)
export_hf_checkpoint(pipe.text_encoder, export_dir="./zimage-fp8/text_encoder/")
```

The exported directory contains `*.safetensors` with `_quantization_metadata` in the header.

### Load Quantized Transformer in ComfyUI

ComfyUI's `quant_ops.py` reads `_quantization_metadata` automatically when a node
loads a `.safetensors` checkpoint. The `MixedPrecisionOps` class in `ops.py` activates
the quantized kernels at model load time when `model_config.layer_quant_config` is present.

```
ComfyUI/models/diffusion_models/
└── zimage-fp8.safetensors   ← place exported checkpoint here

ComfyUI/models/text_encoders/
└── zimage-te-fp8.safetensors

ComfyUI/models/vae/
└── zimage-vae.safetensors   ← use original BF16 VAE (not worth quantizing)
```

Load in a ComfyUI workflow:
```
[Load Diffusion Model]  ← picks up quantization_metadata automatically
        ↓
[ZImage Sampler / KSampler with ZImage model]
        ↓
[VAE Decode]
        ↓
[Save Image]
```

## B.8 Analytical Benchmarking Table

The code cell below derives memory footprints and latency estimates from first principles
using Z-Image's published architecture parameters.
It requires **no model download** — all numbers come from the roofline model.

**Two reference points are computed for each configuration:**
- **H100 SXM5** — native FP8 dispatch via TRT-LLM or ComfyUI (real speedup)
- **DGX Spark / GB10** — PyTorch simulation mode (quantize → dequantize → BF16 GEMM; always slower than BF16, as shown in Appendix A)

> Run the next cell to see the full table.

In [ ]:
"""
Appendix B — Z-Image Benchmarking Table

Part 1: Roofline estimates for H100 SXM5 (from architecture parameters, no model needed).
Part 2: Real measurements on DGX Spark / GB10 (PyTorch simulation mode).
"""
import numpy as np
import pandas as pd
import os, json as _json

# ── Architecture (NextDiT, dim=3840, 48 layers) ────────────────────────────
DIM         = 3840
N_LAYERS    = 48
N_HEADS     = 16
N_KV_HEADS  = 16
HEAD_DIM    = DIM // N_HEADS  # 240
FFN_DIM     = int(2 / 3 * 4 * DIM / 64) * 64  # SwiGLU hidden = 10240

p_per_layer = (DIM * (3 * HEAD_DIM * N_HEADS)        # qkv
             + (HEAD_DIM * N_HEADS) * DIM              # out
             + 3 * DIM * FFN_DIM)                      # w1+w2+w3 FFN
p_xfmr      = N_LAYERS * p_per_layer + N_LAYERS * (2 * DIM) + DIM * (2 * 2 * DIM)

P_TE  = 4_000_000_000     # Qwen3-4B text encoder
P_VAE = 80_000_000        # VAE decoder

# Sequence length: VAE ×8 downscale, patch_size=2 → 1024/8/2 = 64 per side
SEQ_LEN_1024 = (1024 // 8 // 2) ** 2   # 4096 tokens at 1024×1024
SEQ_LEN_512  = (512  // 8 // 2) ** 2   # 1024 tokens at 512×512

# ── H100 SXM5 specs ────────────────────────────────────────────────────────
H100_BW      = 3.35e12    # 3.35 TB/s HBM3
H100_FP8_TF  = 1979e12    # FP8 tensor core FLOP/s
H100_BF16_TF = 989.5e12   # BF16 tensor core FLOP/s

# ── Precision bytes per weight ─────────────────────────────────────────────
BF16 = 2; FP8 = 1; INT8 = 1; INT4 = 0.5

def vram_gb(x_bpp, te_bpp, kv_bpp, seq_len=SEQ_LEN_1024):
    kv = 2 * N_LAYERS * N_KV_HEADS * HEAD_DIM * seq_len * kv_bpp
    return (p_xfmr * x_bpp + P_TE * te_bpp + P_VAE * BF16 + kv) / 1e9

def latency_h100(x_bpp, te_bpp, kv_bpp, n_steps=50, seq_len=SEQ_LEN_1024):
    """Roofline: max(compute-bound, memory-bound) for transformer + one TE pass."""
    tflops = H100_FP8_TF if x_bpp <= FP8 else H100_BF16_TF
    # Transformer: FLOPs ≈ 2 × weights × seq_len per step
    flops_step = 2.0 * p_per_layer * N_LAYERS * seq_len
    bytes_step  = p_per_layer * N_LAYERS * x_bpp
    kv_bytes    = 2 * N_LAYERS * N_KV_HEADS * HEAD_DIM * seq_len * kv_bpp
    t_xfmr = max(flops_step * n_steps / tflops,
                 (bytes_step + kv_bytes) * n_steps / H100_BW)
    # Text encoder: one pass, 256-token prompt
    te_tf = H100_FP8_TF if te_bpp <= FP8 else H100_BF16_TF
    t_te  = max(2.0 * P_TE * 256 / te_tf, P_TE * te_bpp / H100_BW)
    return t_xfmr + t_te

# ── Table A: H100 roofline estimates ──────────────────────────────────────
kv_bf16_mb = 2 * N_LAYERS * N_KV_HEADS * HEAD_DIM * SEQ_LEN_1024 * BF16 / 1e6
kv_fp8_mb  = kv_bf16_mb / 2

print(f"Z-Image architecture: NextDiT {p_xfmr/1e9:.1f}B params | Qwen3-4B TE | seq_len={SEQ_LEN_1024} @ 1024px")
print(f"KV cache at 1024×1024: BF16={kv_bf16_mb:.0f} MB → FP8={kv_fp8_mb:.0f} MB")

configs = [
    ("BF16 baseline",        BF16, BF16, BF16),
    ("FP8 transformer",      FP8,  BF16, BF16),
    ("FP8 xfmr + FP8 KV",   FP8,  BF16, FP8 ),
    ("FP8 xfmr + INT8 TE",  FP8,  INT8, BF16),
    ("Full FP8 stack",       FP8,  FP8,  FP8 ),
    ("INT4-AWQ transformer", INT4, BF16, BF16),
]

bm  = vram_gb(BF16, BF16, BF16)
bh  = latency_h100(BF16, BF16, BF16)

rows_h100 = []
for name, x, t, k in configs:
    m = vram_gb(x, t, k);  h = latency_h100(x, t, k)
    rows_h100.append({
        "Configuration"    : name,
        "Total VRAM (GB)"  : f"{m:.1f}",
        "Mem reduction"    : f"{bm/m:.2f}x",
        "H100 Latency (s)" : f"{h:.1f}",
        "H100 Speedup"     : f"{bh/h:.2f}x",
    })

df_h100 = pd.DataFrame(rows_h100).set_index("Configuration")

print("\n── Table A: H100 SXM5 roofline estimates (50 steps @ 1024×1024) ───")
try:
    from IPython.display import display; display(df_h100)
except Exception:
    pd.set_option("display.max_colwidth", 25)
    pd.set_option("display.width", 110)
    print(df_h100.to_string())

# ── Table B: Measured on DGX Spark / GB10 ─────────────────────────────────
real_data = {
    "BF16 baseline":      {"latency_s": 32.34, "mem_mb": 20712},
    "FP8 transformer":    {"latency_s": 42.78, "mem_mb": 20711},
    "FP8 xfmr + FP8 KV": {"latency_s": 42.12, "mem_mb": 20711},
    "FP8 xfmr + INT8 TE":{"latency_s": 49.81, "mem_mb": 20719},
    "Full FP8 stack":     {"latency_s": 42.47, "mem_mb": 20712},
}
_jpath = "/workspace/model-opt/zimage_bench_results.json"
if os.path.exists(_jpath):
    with open(_jpath) as _f:
        real_data = _json.load(_f)

bl_lat = real_data["BF16 baseline"]["latency_s"]
N_STEPS_REAL = 20

rows_gb10 = []
for name, r in real_data.items():
    lat = r["latency_s"]
    ok  = lat <= bl_lat
    rows_gb10.append({
        "Configuration"   : name,
        "Mem (MB)"        : f"{r['mem_mb']:.0f}",
        "Latency (s)"     : f"{lat:.2f}",
        "Steps/sec"       : f"{N_STEPS_REAL/lat:.2f}",
        "vs BF16"         : f"{bl_lat/lat:.2f}x {'✓' if ok else '✗ sim'}",
    })

df_gb10 = pd.DataFrame(rows_gb10).set_index("Configuration")

print("\n── Table B: Measured on DGX Spark / GB10  (20 steps @ 512×512) ────")
try:
    from IPython.display import display; display(df_gb10)
except Exception:
    print(df_gb10.to_string())

print()
print("Key observations:")
print("  1. Memory (Table B): Simulation mode stores weights in BF16 internally;")
print("     actual VRAM savings only appear when the model is exported + loaded natively")
print("     (e.g., export_hf_checkpoint → ComfyUI). This is why all configs show ~20 GB.")
print("  2. Latency (Table B): Quantized configs are 1.3–1.5× SLOWER on GB10 due to")
print("     PyTorch simulation overhead (quant→dequant→BF16 GEMM).")
print("     ⚠ This is NOT the same as the 13.5× GEMM test in Appendix A — that measured")
print("     raw GEMM overhead. In the full pipeline, non-GEMM ops (attention, norm, VAE)")
print("     dilute the simulation cost to ~1.3× end-to-end.")
print("  3. H100 speedup (Table A): FP8 and INT4 GEMM are dispatched natively; the roofline")
print("     predicts ~2× speedup and ~2× VRAM reduction for the Full FP8 stack.")
print("  4. Full FP8 stack at 1024×1024: ~14 GB total VRAM — fits in a single 16 GB GPU.")
